# Boop — ThompsonZero Training

An overhaul of the KataZero notebook (`boop_kataboop_training.ipynb`) that replaces
PUCT-on-a-policy-network with **Thompson sampling over per-action Beta
posteriors**. Uses the same fixed Boop rules as the KataZero notebook
(mixed kitten+cat lines graduate; one uniformly-random qualifying line resolves
per pass — rulebook p.4 / fig.4).

Boop has no draws — someone always graduates toward 3 cats — so the network
predicts exactly **two values per action**: a win probability and a confidence
(the prior's worth in visits). The prior for action `a` is
`Beta(conf·p_win, conf·(1−p_win))`. (The engine's 500-move cap is a dead rule
in practice; if it ever fires, that terminal counts as half a win + half a loss.)

| Component | KataZero | ThompsonZero (this notebook) |
|---|---|---|
| NN output | policy logits + scalar value | per action: **win prob + confidence α₀** → prior `Beta(α₀·p, α₀·(1−p))` |
| In-tree selection | PUCT (prior × visit bonus) | **Thompson sampling**: sample `p ~ Beta(α)` per action, play the argmax |
| Exploration noise | root Dirichlet noise + temperature | **none needed** — posterior sampling explores by itself |
| Backup | running mean of scalar values | **conjugate count update**: observe win/loss → `α[outcome] += 1` along the path (perspective flips every ply) |
| Leaf evaluation | value-head scalar | expand leaf with NN priors, Thompson-sample its actions, then **draw a Bernoulli win/loss** from the winning sample (the "rollout result") |
| Move selection | visit-count softmax w/ temperature | **Thompson-sample the root posteriors** (first `TEMP_THRESHOLD` moves), then posterior-mean argmax |
| Training target | visit distribution + z/root-value mix | the **root's posterior Beta per action** (prior counts + observed counts) |
| Loss | CE + MSE + entropy bonus | **KL( Beta(posterior) ‖ Beta(predicted prior) )** — one closed-form term, no value head |
| Virtual loss | pretend the path lost (−1 value) | pretend the path lost (**+1 on the loss pseudo-count**) |

### The algorithm in one paragraph
Each edge `(s, a)` holds a Beta distribution over that action's outcome
(win, loss) *from the perspective of the player taking it*. The network
provides the prior `α = α₀ · (p_win, 1−p_win)`; every simulation that passes
through the edge adds one count to the outcome it observed (flipped to the
mover's perspective). Selection at every node — and the final move choice at
the root — is Thompson sampling: draw `p ~ Beta(α)` for each action and take
the argmax. Uncertain actions (small α₀) produce wide samples and get
explored; confidently-bad actions almost never win the argmax. When a
simulation reaches an unexpanded node it is expanded with NN priors, its
actions are Thompson-sampled once, and a Bernoulli win/loss is drawn from the
winning sample — that outcome is what gets backed up. After `N` simulations
the root's per-action posteriors become the training targets, and the network
is trained to minimize the Beta-to-Beta KL divergence between the MCTS
posterior and its predicted prior.

### Network architecture
```
Input (9, 6, 6)  →  Stem (Conv 3×3, GroupNorm, ReLU)
                 →  N × ResBlock (Conv-GN-ReLU-Conv-GN + SE) + skip
                 →  Head: 1×1 conv (4 ch) → flatten
                       → Linear(108) → sigmoid  → p_win per action
                       → Linear(108) → softplus → confidence α₀ ∈ [CONF_MIN, CONF_MAX]
```

### Engineering kept from the KataZero notebook
- Vectorized Boop engine (with the kata_update rule fixes), 8× symmetry
  augmentation, playout-cap randomization
- Batched-leaf waves (Thompson sampling is stochastic, so waves are naturally
  diverse; a loss-count virtual loss still keeps duplicate leaves rare)
- **Full MCTS-Solver**: proven wins/losses propagate up the tree (not just
  immediate terminals); the search stops early once the root is proven, and the
  proven values are written into the training target with max confidence
- **Opponent-pool diversity**: a fraction of self-play games are played against
  a frozen benchmark checkpoint (one random side) instead of the live net
  mirror-matching itself, to curb non-transitive strategy cycling; only the live
  side's moves become training samples
- Multiprocess self-play workers + central GPU inference server (DirectML-friendly)
- Running-Elo benchmark tournaments in independent tracks (policy / MCTS-25 / MCTS-50)
- Checkpointing + resume

The only *methodological* differences from KataZero are the two the algorithm
requires: the NN output (win-prob + confidence Beta vs policy + value) and how
in-tree/root moves are chosen (Thompson sampling vs PUCT). KataZero's
noise-repair machinery (root Dirichlet noise, forced playouts, policy-target
pruning) has no analogue here because Thompson sampling needs no injected noise.

### Arena
The last cells load any **original KataZero checkpoint** (policy+value net,
PUCT search) and pit it against the ThompsonZero net — policy-only and with
search — so the two training regimes can be compared head-to-head.

In [ ]:
%pip install open_spiel -q
# AMD/Intel GPU acceleration via DirectML (Windows / WSL2). Needs Python
# 3.11 or 3.12 (torch-directml has no 3.13 wheel) and pins torch==2.3.1.
# Harmless if you're on CUDA/CPU — the device code below falls back.
%pip install torch-directml -q

In [ ]:
# Boop board game -- inlined so Colab works without the custom package.
# Faithful rules: graduation conserves pieces (kittens BECOME cats), and a
# player whose pool is empty must graduate a kitten of their choice rather than
# being stuck (which previously caused spurious draws).
# Perf: win checks, promotion, legal moves and the observer are vectorized over
# a precomputed 3-in-a-row line table, and clone() is a hand-rolled field copy
# instead of deepcopy — MCTS clones a state once per simulation, so these are
# the hottest paths in the whole pipeline.

import numpy as np
from open_spiel.python.observation import IIGObserverForPublicInfoGame
import pyspiel

_NUM_PLAYERS = 2
_ROWS = 6
_COLS = 6
_NUM_CELLS = _ROWS * _COLS
_NUM_PIECE_TYPES = 2
_NUM_PIECES = 8                              # each player has exactly 8 pieces
_GRADUATE_OFFSET = _NUM_PIECE_TYPES * _NUM_CELLS  # 72: graduation actions start here
_NUM_ACTIONS = _GRADUATE_OFFSET + _NUM_CELLS      # 108 = 72 placement + 36 graduate
_MAX_KITTENS = 8
_MAX_CATS = 8                               # all 8 pieces can become cats
_MAX_GAME_LENGTH = 500

_EMPTY = 0
_P0_KITTEN = 1
_P0_CAT = 2
_P1_KITTEN = 3
_P1_CAT = 4

_KITTEN_VAL = [_P0_KITTEN, _P1_KITTEN]
_CAT_VAL = [_P0_CAT, _P1_CAT]
_PIECE_VALS = [[_P0_KITTEN, _P0_CAT], [_P1_KITTEN, _P1_CAT]]

# Every 3-in-a-row line on the board (horizontal, vertical, both diagonals) as
# indices into the flattened 36-cell board: shape (80, 3). Win detection and
# promotion become single vectorized gathers instead of triple Python loops.
_LINE_IDX = np.array(
    [[(r + k * dr) * _COLS + (c + k * dc) for k in range(3)]
     for r in range(_ROWS) for c in range(_COLS)
     for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1))
     if 0 <= r + 2 * dr < _ROWS and 0 <= c + 2 * dc < _COLS],
    dtype=np.int64)

_GAME_TYPE = pyspiel.GameType(
    short_name='python_boop',
    long_name='Python Boop',
    dynamics=pyspiel.GameType.Dynamics.SEQUENTIAL,
    chance_mode=pyspiel.GameType.ChanceMode.DETERMINISTIC,
    information=pyspiel.GameType.Information.PERFECT_INFORMATION,
    utility=pyspiel.GameType.Utility.ZERO_SUM,
    reward_model=pyspiel.GameType.RewardModel.TERMINAL,
    max_num_players=_NUM_PLAYERS,
    min_num_players=_NUM_PLAYERS,
    provides_information_state_string=True,
    provides_information_state_tensor=False,
    provides_observation_string=True,
    provides_observation_tensor=True,
    parameter_specification={})

_GAME_INFO = pyspiel.GameInfo(
    num_distinct_actions=_NUM_ACTIONS,
    max_chance_outcomes=0,
    num_players=_NUM_PLAYERS,
    min_utility=-1.0,
    max_utility=1.0,
    utility_sum=0.0,
    max_game_length=_MAX_GAME_LENGTH)


class BoopGame(pyspiel.Game):
  def __init__(self, params=None):
    super().__init__(_GAME_TYPE, _GAME_INFO, params or dict())

  def new_initial_state(self):
    return BoopState(self)

  def make_py_observer(self, iig_obs_type=None, params=None):
    if ((iig_obs_type is None) or
        (iig_obs_type.public_info and not iig_obs_type.perfect_recall)):
      return BoopObserver(params)
    return IIGObserverForPublicInfoGame(iig_obs_type, params)


class BoopState(pyspiel.State):
  def __init__(self, game):
    super().__init__(game)
    self._game_ref = game
    self._cur_player = 0
    self._is_terminal = False
    self._winner = None
    self._move_count = 0
    self.board = np.zeros((_ROWS, _COLS), dtype=np.int8)
    self._hand = [[_MAX_KITTENS, 0], [_MAX_KITTENS, 0]]

  def clone(self):
    # Fast clone: MCTS calls this once per simulation. Copying the handful of
    # fields directly is ~10x cheaper than the default deepcopy-based clone.
    cp = BoopState(self._game_ref)
    cp._cur_player  = self._cur_player
    cp._is_terminal = self._is_terminal
    cp._winner      = self._winner
    cp._move_count  = self._move_count
    cp.board        = self.board.copy()
    cp._hand        = [self._hand[0][:], self._hand[1][:]]
    return cp

  def current_player(self):
    return pyspiel.PlayerId.TERMINAL if self._is_terminal else self._cur_player

  def _legal_actions(self, player):
    # Forced-graduation rule: if the pool is empty, the player must graduate one
    # of their kittens on the board (returning it to the pool as a cat) instead
    # of placing. Graduation actions are _GRADUATE_OFFSET + cell.
    flat = self.board.reshape(-1)
    hk, hc = self._hand[player]
    if hk == 0 and hc == 0:
      kittens = np.flatnonzero(flat == _KITTEN_VAL[player])
      return (_GRADUATE_OFFSET + kittens).tolist()
    empty = np.flatnonzero(flat == _EMPTY)
    if hk > 0 and hc > 0:
      return np.concatenate([empty, _NUM_CELLS + empty]).tolist()
    if hk > 0:
      return empty.tolist()
    return (_NUM_CELLS + empty).tolist()

  def _apply_action(self, action):
    p = self._cur_player
    if action >= _GRADUATE_OFFSET:
      # Forced graduation: a kitten on the board becomes a cat in the pool.
      cell = action - _GRADUATE_OFFSET
      r, c = cell // _COLS, cell % _COLS
      self.board[r, c] = _EMPTY
      self._hand[p][1] += 1
      self._move_count += 1
      self._post_move(p)        # no piece placed → no boop
      return
    piece_type = action // _NUM_CELLS
    cell = action % _NUM_CELLS
    r, c = cell // _COLS, cell % _COLS
    self._hand[p][piece_type] -= 1
    self.board[r, c] = _PIECE_VALS[p][piece_type]
    self._boop(r, c, is_cat=(piece_type == 1))
    self._move_count += 1
    self._post_move(p)

  def _post_move(self, p):
    """Shared post-move resolution: win checks, promotion, turn handoff."""
    if self._move_count >= _MAX_GAME_LENGTH:
      self._is_terminal = True
      return
    for player in (p, 1 - p):
      if self._check_win(player):
        self._is_terminal = True
        self._winner = player
        return
    self._promote_kittens(p)
    self._promote_kittens(1 - p)
    for player in (p, 1 - p):
      if self._check_win(player):
        self._is_terminal = True
        self._winner = player
        return
    self._cur_player = 1 - p
    # With forced graduation a player is never permanently stuck: an empty pool
    # forces a graduation, and a board of eight cats is already a win. The guard
    # below is defensive only and should not trigger in normal play.
    if not self._legal_actions(self._cur_player):
      self._is_terminal = True

  def _action_to_string(self, player, action):
    if action >= _GRADUATE_OFFSET:
      cell = action - _GRADUATE_OFFSET
      r, c = cell // _COLS, cell % _COLS
      return f'p{player}:graduate@({r},{c})'
    pt = action // _NUM_CELLS
    cell = action % _NUM_CELLS
    r, c = cell // _COLS, cell % _COLS
    piece = 'cat' if pt else 'kitten'
    return f'p{player}:{piece}@({r},{c})'

  def is_terminal(self):
    return self._is_terminal

  def returns(self):
    if self._winner == 0:
      return [1.0, -1.0]
    if self._winner == 1:
      return [-1.0, 1.0]
    return [0.0, 0.0]

  def __str__(self):
    syms = {
        _EMPTY: '.', _P0_KITTEN: 'k', _P0_CAT: 'K',
        _P1_KITTEN: 'o', _P1_CAT: 'O',
    }
    rows = [
        ''.join(syms[self.board[r, c]] for c in range(_COLS))
        for r in range(_ROWS)
    ]
    rows.append(
        f'P0: {self._hand[0][0]}k {self._hand[0][1]}K  '
        f'P1: {self._hand[1][0]}k {self._hand[1][1]}K  '
        f'move={self._move_count}')
    return '\n'.join(rows)

  def _boop(self, r, c, is_cat):
    board = self.board
    for dr in (-1, 0, 1):
      for dc in (-1, 0, 1):
        if dr == 0 and dc == 0:
          continue
        nr, nc = r + dr, c + dc
        if not (0 <= nr < _ROWS and 0 <= nc < _COLS):
          continue
        neighbor = board[nr, nc]
        if neighbor == _EMPTY:
          continue
        neighbor_is_cat = neighbor == _P0_CAT or neighbor == _P1_CAT
        if not is_cat and neighbor_is_cat:
          continue
        dest_r, dest_c = nr + dr, nc + dc
        owner = 0 if (neighbor == _P0_KITTEN or neighbor == _P0_CAT) else 1
        n_type = 1 if neighbor_is_cat else 0
        if not (0 <= dest_r < _ROWS and 0 <= dest_c < _COLS):
          board[nr, nc] = _EMPTY
          self._hand[owner][n_type] += 1
        elif board[dest_r, dest_c] == _EMPTY:
          board[dest_r, dest_c] = neighbor
          board[nr, nc] = _EMPTY

  def _promote_kittens(self, player):
    # Faithful rule (rulebook p.4): a line of 3 of the player's own pieces —
    # kittens AND/OR cats mixed — graduates. Every kitten in the line becomes
    # a cat; every cat in the line simply returns to the pool; either way all
    # three board cells clear and the pool gains 3 cats (pieces conserved).
    # A pure 3-cats-in-a-row is a WIN, checked before this runs, so it never
    # reaches here as a live case.
    kv, cv = _KITTEN_VAL[player], _CAT_VAL[player]
    flat = self.board.reshape(-1)
    while True:
      mine = (flat == kv) | (flat == cv)
      if int(mine.sum()) < 3:
        return
      full = mine[_LINE_IDX].all(axis=1)
      if not full.any():
        return
      # Resolve ONE qualifying line per pass, chosen UNIFORMLY AT RANDOM
      # among all lines that currently qualify (not the player's choice —
      # that would need a new action type, which isn't worth the added
      # action-space complexity). Clearing the chosen line's cells
      # invalidates any OVERLAPPING line, so it is not picked again this
      # call — matching "choose one, leave the rest" for a connected run of
      # 4+ (fig.4). An independent (non-overlapping) line elsewhere still
      # qualifies and is resolved on the loop's next pass.
      candidates = _LINE_IDX[full]
      line = candidates[np.random.randint(len(candidates))]
      flat[line] = _EMPTY
      self._hand[player][1] += 3

  def _check_win(self, player):
    flat = self.board.reshape(-1)
    cats = flat == _CAT_VAL[player]
    n = int(cats.sum())
    if n >= _NUM_PIECES:        # win condition 1: all eight pieces are cats
      return True
    if n < 3:
      return False
    # Win condition 2: three cats in a row (orthogonal or diagonal).
    return bool(cats[_LINE_IDX].all(axis=1).any())


class BoopObserver:
  def __init__(self, params):
    if params:
      raise ValueError(f'Observation parameters not supported; passed {params}')
    board_size = 5 * _ROWS * _COLS
    self.tensor = np.zeros(board_size + 4, np.float32)
    self.dict = {
        'observation': np.reshape(self.tensor[:board_size], (5, _ROWS, _COLS)),
        'hand': self.tensor[board_size:],
    }

  def set_from(self, state, player):
    self.tensor.fill(0)
    obs = self.dict['observation']
    hand = self.dict['hand']
    b = state.board
    opp = 1 - player
    obs[0][b == _EMPTY] = 1.0
    obs[1][b == _KITTEN_VAL[player]] = 1.0
    obs[2][b == _CAT_VAL[player]] = 1.0
    obs[3][b == _KITTEN_VAL[opp]] = 1.0
    obs[4][b == _CAT_VAL[opp]] = 1.0
    hand[0] = state._hand[player][0] / _MAX_KITTENS
    hand[1] = state._hand[player][1] / _MAX_CATS
    hand[2] = state._hand[opp][0] / _MAX_KITTENS
    hand[3] = state._hand[opp][1] / _MAX_CATS

  def string_from(self, state, player):
    del player
    return str(state)


pyspiel.register_game(_GAME_TYPE, BoopGame)

In [ ]:
game = pyspiel.load_game('python_boop')
state = game.new_initial_state()
print('Game:', game.get_type().long_name)
print('Actions:', game.num_distinct_actions())
print('Obs size:', game.observation_tensor_size())
print()
print(state)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import copy
import random

# ── Device selection ──────────────────────────────────────────────────────────
# Same policy as the KataZero notebook: the GPU only wins with BIG batches, so
# self-play NN requests are funneled through one batching server and training
# runs large-batch steps there; small-batch paths (eval, arena) use CPU replicas.
DEVICE_PREFERENCE = 'directml' # 'cpu' | 'directml' | 'cuda' | 'auto'

def _pick_device(pref):
    if pref in ('directml', 'auto'):
        try:
            import torch_directml
            try:    _name = torch_directml.device_name(0)
            except Exception: _name = 'DirectML GPU'
            print(f'Using DirectML: {_name}')
            return torch_directml.device(), 'directml'
        except Exception:
            if pref == 'directml':
                print('DirectML requested but unavailable — falling back.')
    if pref in ('cuda', 'auto') and torch.cuda.is_available():
        return torch.device('cuda'), 'cuda'
    return torch.device('cpu'), 'cpu'

DEVICE, _BACKEND = _pick_device(DEVICE_PREFERENCE)


# ── Outcome convention (used EVERYWHERE in this notebook) ─────────────────────
# Boop has no draws (someone always graduates toward 3 cats; the 500-move cap
# is a dead rule in practice), so every per-action distribution is a BETA over
# (win, loss) FROM THE PERSPECTIVE OF THE PLAYER TAKING THE ACTION, stored as
# its two pseudo-counts. One ply up the tree the mover changes, so an observed
# outcome flips win<->loss (reverse the pair) on the way to the root. Should
# the move cap ever fire, that terminal counts as half a win + half a loss.
_WIN, _LOSS = 0, 1                                 # columns of every (k, 2) array
_TERM_DRAW  = 2                                    # move-cap terminal marker only
_W_VEC = np.array([1.0, 0.0])                      # backup weight vectors
_L_VEC = np.array([0.0, 1.0])
_D_VEC = np.array([0.5, 0.5])
_TERM_VECS = (_W_VEC, _L_VEC, _D_VEC)
# MCTS-Solver: a proven outcome for the mover flips one ply up (win<->loss,
# draw stays), because the parent's mover is the opponent. Indexed by outcome.
_FLIP_TERM = np.array([_LOSS, _WIN, _TERM_DRAW], dtype=np.int8)

ALPHA_FLOOR = 0.05    # per-component Beta floor: keeps sampling + lgamma finite
CONF_MIN    = 0.5     # smallest predictable prior strength (pseudo-observations)
CONF_MAX    = 100.0   # cap: stops confidence from running away across generations


# ── Input helpers ─────────────────────────────────────────────────────────────

def _obs_to_9ch(obs_np):
    """Flat 184-float obs → (9, 6, 6): 5 board planes + 4 hand scalars broadcast."""
    board = obs_np[..., :180].reshape(*obs_np.shape[:-1], 5, 6, 6)
    hand  = obs_np[..., 180:]
    hand_planes = np.broadcast_to(
        hand[..., None, None], hand.shape + (6, 6)).copy()
    return np.concatenate([board, hand_planes], axis=-3)   # (..., 9, 6, 6)


def state_to_tensor(state, device):
    """Single game state → (1, 9, 6, 6) float tensor."""
    obs = np.array(state.observation_tensor(state.current_player()), dtype=np.float32)
    x   = _obs_to_9ch(obs)[None]
    return torch.from_numpy(x).to(device)


def batch_to_tensor(obs_list, device):
    """List of flat 184-float observations → (B, 9, 6, 6) float tensor."""
    obs = np.stack(obs_list).astype(np.float32)
    x   = _obs_to_9ch(obs)
    return torch.from_numpy(x).to(device)


# ── Network modules (identical to the KataZero notebook, incl. state-dict keys,
#    so original checkpoints load into KataNet below) ──────────────────────────

class _GroupNorm(nn.Module):
    """GroupNorm from elementwise ops — DirectML-safe (the fused kernel's
    backward is broken there) and keeps NO running stats (train == eval)."""
    def __init__(self, num_groups, num_channels, eps=1e-5):
        super().__init__()
        self.num_groups = num_groups
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(num_channels))
        self.bias = nn.Parameter(torch.zeros(num_channels))

    def forward(self, x):
        n, c = x.shape[0], x.shape[1]
        xg = x.reshape(n, self.num_groups, -1)
        mean = xg.mean(dim=2, keepdim=True)
        var = (xg - mean).pow(2).mean(dim=2, keepdim=True)
        xg = (xg - mean) / torch.sqrt(var + self.eps)
        x = xg.reshape(x.shape)
        return x * self.weight.view(1, c, 1, 1) + self.bias.view(1, c, 1, 1)


def _norm(channels):
    groups = min(8, channels)
    while channels % groups != 0:
        groups -= 1
    return _GroupNorm(groups, channels)


def _softplus(x):
    """Numerically-stable softplus from elementary ops only. torch's fused
    aten::softplus has no DirectML kernel; relu/abs/exp/log/add all do (softmax
    already exercises exp+log on DML). Identical to F.softplus:
    log(1+e^x) = max(x,0) + log(1 + e^-|x|)."""
    return torch.relu(x) + torch.log(1.0 + torch.exp(-torch.abs(x)))


class SEBlock(nn.Module):
    """Squeeze-and-Excitation channel attention (KataGo-style)."""
    def __init__(self, channels, reduction=4):
        super().__init__()
        mid = max(channels // reduction, 4)
        self.fc = nn.Sequential(
            nn.Linear(channels, mid),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels * 2),
        )

    def forward(self, x):
        s = x.mean(dim=(2, 3))
        scale, bias = self.fc(s).chunk(2, dim=1)
        scale = torch.sigmoid(scale)
        return (x * scale[:, :, None, None]
                  + bias[:, :, None, None])


class ResBlock(nn.Module):
    """Residual block: Conv-GN-ReLU-Conv-GN + SE attention + skip."""
    def __init__(self, channels, use_se=True):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            _norm(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            _norm(channels),
        )
        self.se  = SEBlock(channels) if use_se else nn.Identity()
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.se(self.net(x)) + x)


class ThompsonNet(nn.Module):
    """
    ThompsonZero network for Boop: exactly TWO values per action.

    Input  : (B, 9, 6, 6) — 5 board planes + 4 hand scalars broadcast
    Body   : Conv stem → N × ResBlock(channels, SE)
    Head   : 1×1 conv (4 ch) → flatten →
               win_out  Linear(144, 108) → sigmoid  → p_win
               conf_out Linear(144, 108) → softplus → α₀ ∈ [CONF_MIN, CONF_MAX]
                        (the prior's worth in visits/pseudo-observations)

    forward(x) → (p_win (B, 108), conf (B, 108)).
    The predicted prior for action a is Beta(conf·p_win, conf·(1−p_win)).
    """
    def __init__(self, channels=128, num_blocks=6):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(9, channels, 3, padding=1, bias=False),
            _norm(channels),
            nn.ReLU(inplace=True),
        )
        self.body = nn.Sequential(*[ResBlock(channels) for _ in range(num_blocks)])
        self.head = nn.Sequential(
            nn.Conv2d(channels, 4, 1, bias=False),
            _norm(4),
            nn.ReLU(inplace=True),
            nn.Flatten(),
        )
        self.win_out  = nn.Linear(4 * 6 * 6, _NUM_ACTIONS)
        self.conf_out = nn.Linear(4 * 6 * 6, _NUM_ACTIONS)
        # Start near p_win = 0.5 with a weak prior (α₀ ≈ 2.4): an untrained net
        # should claim LOW confidence so search observations dominate the
        # posterior from the very first generation.
        nn.init.constant_(self.conf_out.bias, 1.4)   # CONF_MIN + softplus(1.4) ≈ 2.4

    def forward(self, x):
        h     = self.head(self.body(self.stem(x)))
        p_win = torch.sigmoid(self.win_out(h))          # sigmoid: DML-proven (SE)
        conf  = torch.clamp(CONF_MIN + _softplus(self.conf_out(h)), max=CONF_MAX)
        return p_win, conf


class KataNet(nn.Module):
    """The ORIGINAL KataZero policy+value network (exact module structure of
    BoopNet in boop_kataboop_training.ipynb, so its checkpoints load directly).
    Used only by the arena cells at the bottom of this notebook."""
    def __init__(self, channels=128, num_blocks=6):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(9, channels, 3, padding=1, bias=False),
            _norm(channels),
            nn.ReLU(inplace=True),
        )
        self.body = nn.Sequential(*[ResBlock(channels) for _ in range(num_blocks)])

        self.policy_head = nn.Sequential(
            nn.Conv2d(channels, 2, 1, bias=False),
            _norm(2),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(2 * 6 * 6, _NUM_ACTIONS),
        )
        self.value_head = nn.Sequential(
            nn.Conv2d(channels, 1, 1, bias=False),
            _norm(1),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(36, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 1),
            nn.Tanh(),
        )

    def forward(self, x):
        x = self.body(self.stem(x))
        return self.policy_head(x), self.value_head(x)


print(f'Device: {DEVICE}')
_demo = ThompsonNet()
print(f'ThompsonNet params: {sum(p.numel() for p in _demo.parameters()):,}')
del _demo

In [ ]:
# ── 8-fold board symmetry augmentation ─────────────────────────────────────────
# The 6×6 Boop board has 8 dihedral symmetries (4 rotations × 2 reflections).
# Every self-play sample is augmented with all 8 copies for free training data.

def _compute_action_perm(sym_idx):
    """
    Precompute the 108-element action permutation for board symmetry sym_idx:
    perm[a] is where original action a LANDS on the transformed board.
    Symmetries 0-3: 0°/90°/180°/270° CCW rotation.
    Symmetries 4-7: horizontal flip followed by the same rotations.
    """
    k    = sym_idx % 4
    flip = sym_idx >= 4
    perm = np.empty(_NUM_ACTIONS, dtype=np.int32)
    for pt in range(2):
        for r in range(6):
            for c in range(6):
                nr, nc = r, c
                if flip:
                    nc = 5 - nc
                for _ in range(k):       # k × 90° CCW: (r,c) → (5-c, r)
                    nr, nc = 5 - nc, nr
                perm[pt * 36 + r * 6 + c] = pt * 36 + nr * 6 + nc
    for r in range(6):                   # graduation actions (cell-indexed)
        for c in range(6):
            nr, nc = r, c
            if flip:
                nc = 5 - nc
            for _ in range(k):
                nr, nc = 5 - nc, nr
            perm[_GRADUATE_OFFSET + r * 6 + c] = _GRADUATE_OFFSET + nr * 6 + nc
    return perm

_ACTION_PERMS = [_compute_action_perm(i) for i in range(8)]


def augment_sample(obs_flat, target):
    """All 8 symmetry copies of an (obs, per-action Beta target) sample.
    `target` is (108, 2); rows are SCATTERED with the forward permutation
    (aug[perm[a]] = target[a]) so the target moves the same direction as the
    board. (np.rot90 sends cell (r,c) to (5-c,r) — the same map as perm — so
    gathering with perm would rotate the target the wrong way for the two
    non-involutive symmetries, rot90 and rot270.)"""
    obs   = np.asarray(obs_flat, dtype=np.float32)
    board = obs[:180].reshape(5, 6, 6)
    hand  = obs[180:]                       # hand counts: spatially invariant
    tgt   = np.asarray(target, dtype=np.float32)

    out = []
    for sym_idx in range(8):
        k    = sym_idx % 4
        flip = sym_idx >= 4
        b    = board.copy()
        if flip:
            b = b[:, :, ::-1]             # horizontal flip (cols)
        b = np.rot90(b, k=k, axes=(1, 2)) # k × 90° CCW
        aug_obs = np.concatenate([b.reshape(-1).copy(), hand])
        aug_tgt = np.zeros_like(tgt)
        aug_tgt[_ACTION_PERMS[sym_idx]] = tgt
        out.append((aug_obs, aug_tgt))
    return out


# ── Thompson-MCTS tree ─────────────────────────────────────────────────────────
# Each node stores, for every legal action, a Beta posterior over that action's
# outcome (win, loss) from the MOVER's perspective:
#   alphas[i] = NN prior (conf · [p_win, 1−p_win]) + observed counts so far.
# Selection = Thompson sampling; backup = +1 on the observed outcome component.
# Virtual loss = a temporary +1 on the LOSS component of in-flight edges, so a
# batched wave spreads out instead of re-picking the same leaf.

class _TNode:
    """One expanded state. `term[i] >= 0` marks a PROVEN outcome for taking
    edge i (from this node's mover's perspective): either the edge lands on a
    terminal state, or MCTS-Solver has proven the whole subtree below it."""
    __slots__ = ('player', 'legal', 'alphas', 'vloss', 'term', 'children')

    def __init__(self, player, legal, pwin_row, conf_row):
        self.player   = player
        self.legal    = np.asarray(legal, dtype=np.int32)
        pw = pwin_row[self.legal].astype(np.float64)
        cf = conf_row[self.legal].astype(np.float64)
        self.alphas   = np.maximum(np.stack([cf * pw, cf * (1.0 - pw)], axis=1),
                                   ALPHA_FLOOR)
        self.vloss    = np.zeros(len(legal), dtype=np.int32)
        self.term     = np.full(len(legal), -1, dtype=np.int8)
        self.children = [None] * len(legal)


def _thompson_values(node, rng, use_vloss):
    """One Thompson sample per action → sampled value p_win − p_loss, with
    known-terminal edges pinned to ±2 (0 for a move-cap draw) so proven wins
    always take the argmax."""
    a = node.alphas
    if use_vloss and node.vloss.any():
        a = a.copy()
        a[:, _LOSS] += node.vloss
    g = rng.standard_gamma(a)
    p = g / (g.sum(axis=1, keepdims=True) + 1e-12)
    v = p[:, _WIN] - p[:, _LOSS]
    t = node.term
    v[t == _WIN]      =  2.0
    v[t == _LOSS]     = -2.0
    v[t == _TERM_DRAW] = 0.0
    return v, p


def _mean_values(node):
    """Posterior-mean value per action (for near-greedy late-game moves/eval)."""
    a  = node.alphas
    a0 = a.sum(axis=1)
    v  = (a[:, _WIN] - a[:, _LOSS]) / a0
    t  = node.term
    v[t == _WIN]      =  2.0
    v[t == _LOSS]     = -2.0
    v[t == _TERM_DRAW] = 0.0
    return v


def _select_leaf(root, root_state, rng):
    """Descend by Thompson sampling until an unexpanded or terminal edge.
    Applies virtual loss along the path. Returns (path, leaf_state, edge):
      terminal edge  → (path, None, None)  — outcome already in node.term
      unexpanded     → (path, state-at-leaf, (node, idx))  — needs NN eval
    path is a list of (node, action_idx)."""
    node  = root
    state = root_state.clone()
    path  = []
    while True:
        v, _ = _thompson_values(node, rng, use_vloss=True)
        idx  = int(v.argmax())
        node.vloss[idx] += 1
        path.append((node, idx))
        if node.term[idx] >= 0:
            return path, None, None
        state.apply_action(int(node.legal[idx]))
        if state.is_terminal():
            r = state.returns()[node.player]
            node.term[idx] = _WIN if r > 0 else (_LOSS if r < 0 else _TERM_DRAW)
            return path, None, None
        child = node.children[idx]
        if child is None:
            return path, state, (node, idx)
        node = child


def _leaf_outcome(leaf, rng):
    """The 'rollout result' at a freshly expanded leaf: Thompson-sample the
    leaf's actions once, then draw a Bernoulli win/loss (for the LEAF's mover)
    from the winning sample's win probability. Returns a backup weight pair."""
    v, p = _thompson_values(leaf, rng, use_vloss=False)
    return _W_VEC if rng.random_sample() < p[int(v.argmax()), _WIN] else _L_VEC


def _backup(path, wvec, outcome_player):
    """Add one observation along the path: `wvec` is the (win, loss) count
    weights from `outcome_player`'s perspective; each edge counts it from its
    own mover's perspective (reversed one ply up). Also removes the virtual
    loss applied at selection."""
    flipped = wvec[::-1]
    for node, idx in path:
        node.vloss[idx] -= 1
        node.alphas[idx] += wvec if node.player == outcome_player else flipped


# ── MCTS-Solver: propagate proven wins/losses up the tree ─────────────────────
# `node.term[i]` is the proven outcome of edge i for node.player. A node itself
# is solved once its BEST attainable outcome is certain: a single proven-WIN
# edge proves the node a win immediately (the mover just takes it); a node is a
# proven loss/draw only when EVERY edge is proven (no unexplored edge could beat
# the current best). When a node becomes solved, the edge that enters it from
# its parent is proven with the flipped outcome (the parent's mover is the
# opponent). Proven edges are pinned to ±2/0 during selection, and `_select_leaf`
# short-circuits on them, so solved subtrees are never re-descended.

def _node_solved_outcome(node):
    """The proven outcome for node.player if the node is solved, else None.
    WIN as soon as any edge is a proven win; LOSS/DRAW only once all edges are
    proven (mover then takes the best available)."""
    t = node.term
    if (t == _WIN).any():
        return _WIN
    if (t >= 0).all():
        return _TERM_DRAW if (t == _TERM_DRAW).any() else _LOSS
    return None


def _propagate_solved(path):
    """Walk leaf→root; whenever a node has become fully solved, prove the parent
    edge that enters it (flipped). Stops at the first not-yet-solved ancestor or
    an already-proven parent edge (both cap the cascade)."""
    for k in range(len(path) - 1, 0, -1):
        out = _node_solved_outcome(path[k][0])
        if out is None:
            break
        parent, pidx = path[k - 1]
        if parent.term[pidx] >= 0:
            break
        parent.term[pidx] = _FLIP_TERM[out]


def _backup_terminal(path):
    """Backup for a path that ended on a proven edge (terminal or solver-proven;
    a move-cap draw counts as half a win + half a loss), then propagate any
    newly-solved nodes up the tree."""
    node, idx = path[-1]
    _backup(path, _TERM_VECS[int(node.term[idx])], node.player)
    _propagate_solved(path)


def root_target(root, conf_cap):
    """Root posterior → (108, 2) training target. Proven edges (MCTS-Solver)
    are overwritten with certain, max-confidence targets so an early-stopped
    search still teaches the proven values; the rest keep their sampled
    posterior. Rows are then rescaled (mean-preserving) to at most `conf_cap`
    total count — caps how certain any single generation can teach the net."""
    a = root.alphas.copy()
    t = root.term
    if (t >= 0).any():
        a[t == _WIN]       = (conf_cap, ALPHA_FLOOR)
        a[t == _LOSS]      = (ALPHA_FLOOR, conf_cap)
        a[t == _TERM_DRAW] = (0.5 * conf_cap, 0.5 * conf_cap)
    s = a.sum(axis=1)
    a *= np.minimum(1.0, conf_cap / np.maximum(s, 1e-9))[:, None]
    tgt = np.zeros((_NUM_ACTIONS, 2), dtype=np.float32)
    tgt[root.legal] = a
    return tgt


def root_pick(root, rng, thompson):
    """Final move choice: Thompson-sample the posteriors (exploratory phase) or
    take the posterior-mean argmax (deterministic endgame / evaluation)."""
    if thompson:
        v, _ = _thompson_values(root, rng, use_vloss=False)
    else:
        v = _mean_values(root)
    return int(root.legal[int(v.argmax())])


# ── Batched-leaf Thompson-MCTS bot ─────────────────────────────────────────────

class ThompsonMCTSBot:
    """mcts_search(state) → root _TNode whose .alphas hold the posterior.
    Collects `batch_size` leaves per wave (virtual loss + the inherent
    randomness of Thompson sampling keep waves diverse) and evaluates them in
    one NN forward pass. Weights are read live from `network`."""

    def __init__(self, game, network, device, max_simulations,
                 batch_size=16, random_state=None):
        self.game            = game
        self.network         = network
        self.device          = device
        self.max_simulations = max_simulations
        self.batch_size      = batch_size
        self._rng            = random_state or np.random.RandomState()

    def _eval_batch(self, states):
        obs_list = [s.observation_tensor(s.current_player()) for s in states]
        x = batch_to_tensor(obs_list, self.device)
        with torch.no_grad():
            p_win, conf = self.network(x)
        return p_win.cpu().numpy(), conf.cpu().numpy()   # (B,108), (B,108)

    def mcts_search(self, state):
        p_win, conf = self._eval_batch([state])
        root = _TNode(state.current_player(), state.legal_actions(),
                      p_win[0], conf[0])
        sims = 0
        while sims < self.max_simulations:
            if _node_solved_outcome(root) is not None:
                break                          # MCTS-Solver: root proven, stop
            wave    = min(self.batch_size, self.max_simulations - sims)
            pending = []                       # (path, leaf_state, (node, idx))
            for _ in range(wave):
                path, st, edge = _select_leaf(root, state, self._rng)
                if st is None:                 # terminal edge: exact outcome
                    _backup_terminal(path)
                    sims += 1
                else:
                    pending.append((path, st, edge))

            # Expand every unique leaf edge in ONE forward pass.
            unique = {}
            for path, st, (node, idx) in pending:
                unique.setdefault((id(node), idx), (node, idx, st))
            if unique:
                entries = list(unique.values())
                pr, cf  = self._eval_batch([st for _, _, st in entries])
                for (node, idx, st), p_row, c_row in zip(entries, pr, cf):
                    node.children[idx] = _TNode(st.current_player(),
                                                st.legal_actions(), p_row, c_row)
            # Draw an outcome at each pending leaf and back it up.
            for path, st, (node, idx) in pending:
                leaf = node.children[idx]
                _backup(path, _leaf_outcome(leaf, self._rng), leaf.player)
                sims += 1
        return root


def make_thompson_bot(game, network, device, num_simulations, batch_size=16):
    return ThompsonMCTSBot(game, network, device, num_simulations,
                           batch_size=batch_size)


# ── Evaluation ─────────────────────────────────────────────────────────────────

def policy_action(network, state, device, sample=False, rng=None):
    """Search-free move from the raw network: argmax of the prior-mean win
    probability, or (sample=True) one Thompson sample from the predicted Beta
    priors — the search-free analogue of temperature sampling."""
    with torch.no_grad():
        p_win, conf = network(state_to_tensor(state, device))
    p_win = p_win[0].cpu().numpy()
    conf  = conf[0].cpu().numpy()
    legal = state.legal_actions()
    if sample:
        rng = rng or np.random
        pw = p_win[legal]
        a  = np.maximum(np.stack([conf[legal] * pw,
                                  conf[legal] * (1.0 - pw)], axis=1),
                        ALPHA_FLOOR)
        g = rng.standard_gamma(a)
        v = g[:, _WIN] / (g.sum(axis=1) + 1e-12)
    else:
        v = p_win[legal]
    return legal[int(v.argmax())]


def play_match(net_a, net_b, game, n_games, device):
    """net_a vs net_b over n_games, alternating colours. net == None ==> random.
    Search-free, Thompson-sampled moves (per-game variety without extra noise).
    Returns (wins_a, draws, wins_b) from net_a's view — the quick progress
    pulse (does not touch Elo)."""
    wa = dd = wb = 0
    for i in range(n_games):
        state = game.new_initial_state()
        a_player = i % 2
        while not state.is_terminal():
            net = net_a if state.current_player() == a_player else net_b
            if net is None:
                action = random.choice(state.legal_actions())
            else:
                action = policy_action(net, state, device, sample=True)
            state.apply_action(action)
        r = state.returns()[a_player]
        if r > 0:   wa += 1
        elif r < 0: wb += 1
        else:       dd += 1
    return wa, dd, wb


def _mcts_move(bot, state):
    """Posterior-mean argmax after a full search (the in-tree Thompson sampling
    gives the per-game variety that keeps a round-robin from replaying
    identical games)."""
    return root_pick(bot.mcts_search(state), bot._rng, thompson=False)


# ── Benchmark-pool evaluation: running-Elo tournaments (no fixed anchor) ─────────
# Identical structure to the KataZero notebook: every net is a tournament
# player entering at START_ELO, updated with the Elo K-factor, in independent
# TRACKS (policy-only and one per MCTS budget) with per-track Elo tables and
# head-to-head matrices.

import math
import itertools
from collections import defaultdict


def run_tournament(players, elos, wdl, game, device,
                   games_per_pair=10, k=32.0, sims=0, opening_plies=0,
                   focus_label=None, refresh_pairs=0):
    """One full round-robin for a single track. Moves are chosen search-free
    (sims == 0: prior-mean argmax) or by Thompson-MCTS with `sims` simulations
    (posterior-mean argmax). The first `opening_plies` moves of each game are
    random (both sides) only to vary games; strength is then measured
    noise-free. Every unordered pair plays `games_per_pair` games, half per
    colour, all shuffled into random order; Elo is updated after each game.
    `elos` must already hold a rating for every label. Mutates `elos`/`wdl`."""
    net  = {p['label']: p['net'] for p in players}
    bots = {}
    if sims > 0:
        bs = max(1, min(sims, 16))
        for p in players:
            if p['net'] is not None:
                bots[p['label']] = make_thompson_bot(game, p['net'], device,
                                                     sims, bs)

    def pick(label, state):
        if net[label] is None:
            return random.choice(state.legal_actions())
        if sims <= 0:
            return policy_action(net[label], state, device, sample=False)
        return _mcts_move(bots[label], state)

    pairs = list(itertools.combinations(net.keys(), 2))
    if focus_label is not None:
        # Linear-cost mode: every pair involving the newest model, plus a few
        # random old-vs-old pairs so earlier ratings keep mixing.
        focus  = [p for p in pairs if focus_label in p]
        others = [p for p in pairs if focus_label not in p]
        random.shuffle(others)
        pairs = focus + others[:refresh_pairs]
    schedule = []
    for a, b in pairs:
        half = games_per_pair // 2
        schedule += [(a, b)] * half
        schedule += [(b, a)] * half
        schedule += [(a, b)] * (games_per_pair - 2 * half)   # odd remainder
    random.shuffle(schedule)

    for p0, p1 in schedule:
        state = game.new_initial_state()
        ply = 0
        while not state.is_terminal():
            if ply < opening_plies:
                action = random.choice(state.legal_actions())  # variety only
            else:
                lab = p0 if state.current_player() == 0 else p1
                action = pick(lab, state)
            state.apply_action(action)
            ply += 1
        r  = state.returns()[0]
        s0 = 1.0 if r > 0 else (0.0 if r < 0 else 0.5)
        e0 = 1.0 / (1.0 + 10 ** ((elos[p1] - elos[p0]) / 400.0))
        elos[p0] += k * (s0 - e0)
        elos[p1] += k * ((1.0 - s0) - (1.0 - e0))
        cell = wdl[(p0, p1)]
        if r > 0:   cell[0] += 1
        elif r < 0: cell[2] += 1
        else:       cell[1] += 1

In [ ]:
# ── Parallel self-play: many games, one forward pass ────────────────────────────
# ThompsonMCTSBot batches leaves within ONE search (≤ its batch_size). The next
# multiplier is batching ACROSS games: run N games concurrently, collect every
# game's leaf wave, and evaluate them all in a single NN forward pass — same
# orchestration as the KataZero notebook, on the Thompson tree ops above.
import os


def _nn_eval_states(network, device, states):
    """States → (p_win (B, 108), conf (B, 108)) in one forward pass."""
    obs_list = [s.observation_tensor(s.current_player()) for s in states]
    x = batch_to_tensor(obs_list, device)
    with torch.no_grad():
        p_win, conf = network(x)
    return p_win.cpu().numpy(), conf.cpu().numpy()


class ThompsonParallelSelfPlay:
    """Interleaves the searches of n_parallel self-play games so their leaf
    waves (and root expansions) share one NN forward pass. Weights are read
    live from `network`, so games in flight pick up training updates
    immediately (standard asynchronous self-play behaviour)."""

    def __init__(self, game, network, device, n_parallel=8, wave_per_game=8,
                 fast_sims=20, full_sims=100, fast_prob=0.75, temp_threshold=30,
                 conf_cap=100.0, seed=None,
                 pool_prob=0.0, checkpoint_dir=None, channels=None,
                 num_blocks=None):
        self.game = game
        self.network = network
        self.device = device
        self.n_parallel = n_parallel
        self.wave = wave_per_game
        self.fast_sims = fast_sims
        self.full_sims = full_sims
        self.fast_prob = fast_prob
        self.temp_threshold = temp_threshold
        self.conf_cap = conf_cap
        self._rng = np.random.RandomState(seed)
        # Opponent diversity (see _resolve_pool_moves): some fraction of games
        # have one side played by a frozen historical checkpoint instead of
        # mirror-matching the live net against itself. Only the live side's
        # moves become training samples.
        self.pool_prob = pool_prob
        self.checkpoint_dir = checkpoint_dir
        self.channels = channels
        self.num_blocks = num_blocks
        self._pool_nets = {}          # benchmark label -> frozen CPU ThompsonNet
        self.slots = [self._new_game() for _ in range(n_parallel)]

    def _load_pool_net(self, label):
        net = self._pool_nets.get(label)
        if net is None:
            path = os.path.join(self.checkpoint_dir, f'bench_{label}.pt')
            net = ThompsonNet(self.channels, self.num_blocks)
            net.load_state_dict(torch.load(path, map_location='cpu',
                                           weights_only=True))
            net.eval()
            self._pool_nets[label] = net
        return net

    def _new_game(self):
        # Playout cap randomization is chosen per game, as in serial self-play.
        sims = (self.fast_sims if self._rng.rand() < self.fast_prob
                else self.full_sims)
        slot = {'state': self.game.new_initial_state(), 'hist': [],
                'move': 0, 'sims': sims, 'root': None, 'n': 0, 'pool': None}
        if self.pool_prob > 0 and self.checkpoint_dir:
            try:
                labels = [f[6:-3] for f in os.listdir(self.checkpoint_dir)
                          if f.startswith('bench_') and f.endswith('.pt')]
            except OSError:
                labels = []
            if labels and self._rng.rand() < self.pool_prob:
                label = labels[self._rng.randint(len(labels))]
                slot['pool'] = {'label': label,
                                'side': int(self._rng.randint(0, 2)),
                                'net': self._load_pool_net(label)}
        return slot

    def _resolve_pool_moves(self):
        """Resolve any pending frozen-opponent moves with a direct forward pass
        + greedy (prior-mean p_win argmax) choice — no search tree, since these
        moves are never training targets. Returns finished episodes (a pool move
        can end the game), matching _step()'s return contract."""
        done = []
        for i, s in enumerate(self.slots):
            pool = s['pool']
            if pool is None:
                continue
            state = s['state']
            if state.current_player() != pool['side']:
                continue
            p_win, _ = _nn_eval_states(pool['net'], 'cpu', [state])
            legal = state.legal_actions()
            action = legal[int(np.argmax(p_win[0][legal]))]
            state.apply_action(action)
            s['move'] += 1
            if state.is_terminal():
                done.append(s['hist'])
                self.slots[i] = self._new_game()
        return done

    def episodes(self):
        """Infinite generator of finished episodes: lists of (obs, target)."""
        while True:
            for data in self._step():
                yield data

    def _step(self):
        """Advance every game by one leaf wave (one joint forward pass);
        returns any episodes that finished this step."""
        rng     = self._rng
        done    = self._resolve_pool_moves()   # frozen-opponent moves first
        pending = []            # (slot_idx, path, (node, idx)) — await NN leaf
        evals   = []            # ('root', i, state) / ('leaf', node, idx, state)
        seen    = set()         # unique leaf edges already queued for eval
        for i, s in enumerate(self.slots):
            pool = s['pool']
            if pool is not None and s['state'].current_player() == pool['side']:
                continue         # next move belongs to the frozen opponent
            if s['root'] is None:
                evals.append(('root', i, None, s['state']))
                continue         # root expands this step; leaves start next step
            if _node_solved_outcome(s['root']) is not None:
                continue         # MCTS-Solver: root proven — finalized below
            wave = min(self.wave, s['sims'] - s['n'])
            for _ in range(max(wave, 0)):
                path, st, edge = _select_leaf(s['root'], s['state'], rng)
                if st is None:
                    _backup_terminal(path)
                    s['n'] += 1
                else:
                    node, idx = edge
                    pending.append((i, path, edge))
                    if (id(node), idx) not in seen:
                        seen.add((id(node), idx))
                        evals.append(('leaf', node, idx, st))

        # ONE forward pass for root expansions + every unique leaf across games.
        if evals:
            p_win, conf = _nn_eval_states(self.network, self.device,
                                          [e[3] for e in evals])
            for (kind, a, b, st), p_row, c_row in zip(evals, p_win, conf):
                child = _TNode(st.current_player(), st.legal_actions(),
                               p_row, c_row)
                if kind == 'root':
                    self.slots[a]['root'] = child
                else:
                    a.children[b] = child
        for i, path, (node, idx) in pending:
            leaf = node.children[idx]
            _backup(path, _leaf_outcome(leaf, rng), leaf.player)
            self.slots[i]['n'] += 1

        # Finalize finished (or solver-proven) searches; emit completed episodes.
        for i, s in enumerate(self.slots):
            if s['root'] is None:
                continue
            if s['n'] < s['sims'] and _node_solved_outcome(s['root']) is None:
                continue
            self._play_move(i)
            if s['state'].is_terminal():
                done.append(s['hist'])
                self.slots[i] = self._new_game()
        return done

    def _play_move(self, i):
        """Turn a finished search into a training sample + a move: the target
        is the root's posterior Beta per action (solver-adjusted for proven
        edges); the move is Thompson-sampled from those posteriors (near-greedy
        posterior mean after temp_threshold — the annealing analogue)."""
        s     = self.slots[i]
        state = s['state']
        obs   = np.asarray(state.observation_tensor(state.current_player()),
                           dtype=np.float32)
        s['hist'].append((obs, root_target(s['root'], self.conf_cap)))
        action = root_pick(s['root'], self._rng,
                           thompson=(s['move'] < self.temp_threshold))
        state.apply_action(action)
        s['move'] += 1
        s['root']  = None
        s['n']     = 0

In [ ]:
# ── Multiprocessing self-play + central GPU inference server ─────────────────
# Same design as the KataZero notebook: N CPU worker processes do ALL game and
# tree work (pure Python — processes sidestep the GIL) and ship their NN
# requests to ONE server thread here, which batches requests from all workers
# into single large forward passes on the GPU. Small-batch inference (eval
# games, tournaments, arena) runs on CPU replicas elsewhere.
#
# Windows/Jupyter constraint: spawned processes cannot pickle notebook-cell
# functions, so the worker code is written to a real module file first.
import pathlib

_WORKER_SRC = r'''# AUTO-GENERATED by boop_thompson_zero_training.ipynb — do not edit by hand.
# Self-play worker module: game engine + Thompson-MCTS tree ops + worker loop.
# Exists as a real file because Windows multiprocessing (spawn) cannot
# pickle functions defined inside notebook cells.

# Boop board game -- inlined so Colab works without the custom package.
# Faithful rules: graduation conserves pieces (kittens BECOME cats), and a
# player whose pool is empty must graduate a kitten of their choice rather than
# being stuck (which previously caused spurious draws).
# Perf: win checks, promotion, legal moves and the observer are vectorized over
# a precomputed 3-in-a-row line table, and clone() is a hand-rolled field copy
# instead of deepcopy — MCTS clones a state once per simulation, so these are
# the hottest paths in the whole pipeline.

import numpy as np
from open_spiel.python.observation import IIGObserverForPublicInfoGame
import pyspiel

_NUM_PLAYERS = 2
_ROWS = 6
_COLS = 6
_NUM_CELLS = _ROWS * _COLS
_NUM_PIECE_TYPES = 2
_NUM_PIECES = 8                              # each player has exactly 8 pieces
_GRADUATE_OFFSET = _NUM_PIECE_TYPES * _NUM_CELLS  # 72: graduation actions start here
_NUM_ACTIONS = _GRADUATE_OFFSET + _NUM_CELLS      # 108 = 72 placement + 36 graduate
_MAX_KITTENS = 8
_MAX_CATS = 8                               # all 8 pieces can become cats
_MAX_GAME_LENGTH = 500

_EMPTY = 0
_P0_KITTEN = 1
_P0_CAT = 2
_P1_KITTEN = 3
_P1_CAT = 4

_KITTEN_VAL = [_P0_KITTEN, _P1_KITTEN]
_CAT_VAL = [_P0_CAT, _P1_CAT]
_PIECE_VALS = [[_P0_KITTEN, _P0_CAT], [_P1_KITTEN, _P1_CAT]]

# Every 3-in-a-row line on the board (horizontal, vertical, both diagonals) as
# indices into the flattened 36-cell board: shape (80, 3). Win detection and
# promotion become single vectorized gathers instead of triple Python loops.
_LINE_IDX = np.array(
    [[(r + k * dr) * _COLS + (c + k * dc) for k in range(3)]
     for r in range(_ROWS) for c in range(_COLS)
     for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1))
     if 0 <= r + 2 * dr < _ROWS and 0 <= c + 2 * dc < _COLS],
    dtype=np.int64)

_GAME_TYPE = pyspiel.GameType(
    short_name='python_boop',
    long_name='Python Boop',
    dynamics=pyspiel.GameType.Dynamics.SEQUENTIAL,
    chance_mode=pyspiel.GameType.ChanceMode.DETERMINISTIC,
    information=pyspiel.GameType.Information.PERFECT_INFORMATION,
    utility=pyspiel.GameType.Utility.ZERO_SUM,
    reward_model=pyspiel.GameType.RewardModel.TERMINAL,
    max_num_players=_NUM_PLAYERS,
    min_num_players=_NUM_PLAYERS,
    provides_information_state_string=True,
    provides_information_state_tensor=False,
    provides_observation_string=True,
    provides_observation_tensor=True,
    parameter_specification={})

_GAME_INFO = pyspiel.GameInfo(
    num_distinct_actions=_NUM_ACTIONS,
    max_chance_outcomes=0,
    num_players=_NUM_PLAYERS,
    min_utility=-1.0,
    max_utility=1.0,
    utility_sum=0.0,
    max_game_length=_MAX_GAME_LENGTH)


class BoopGame(pyspiel.Game):
  def __init__(self, params=None):
    super().__init__(_GAME_TYPE, _GAME_INFO, params or dict())

  def new_initial_state(self):
    return BoopState(self)

  def make_py_observer(self, iig_obs_type=None, params=None):
    if ((iig_obs_type is None) or
        (iig_obs_type.public_info and not iig_obs_type.perfect_recall)):
      return BoopObserver(params)
    return IIGObserverForPublicInfoGame(iig_obs_type, params)


class BoopState(pyspiel.State):
  def __init__(self, game):
    super().__init__(game)
    self._game_ref = game
    self._cur_player = 0
    self._is_terminal = False
    self._winner = None
    self._move_count = 0
    self.board = np.zeros((_ROWS, _COLS), dtype=np.int8)
    self._hand = [[_MAX_KITTENS, 0], [_MAX_KITTENS, 0]]

  def clone(self):
    # Fast clone: MCTS calls this once per simulation. Copying the handful of
    # fields directly is ~10x cheaper than the default deepcopy-based clone.
    cp = BoopState(self._game_ref)
    cp._cur_player  = self._cur_player
    cp._is_terminal = self._is_terminal
    cp._winner      = self._winner
    cp._move_count  = self._move_count
    cp.board        = self.board.copy()
    cp._hand        = [self._hand[0][:], self._hand[1][:]]
    return cp

  def current_player(self):
    return pyspiel.PlayerId.TERMINAL if self._is_terminal else self._cur_player

  def _legal_actions(self, player):
    # Forced-graduation rule: if the pool is empty, the player must graduate one
    # of their kittens on the board (returning it to the pool as a cat) instead
    # of placing. Graduation actions are _GRADUATE_OFFSET + cell.
    flat = self.board.reshape(-1)
    hk, hc = self._hand[player]
    if hk == 0 and hc == 0:
      kittens = np.flatnonzero(flat == _KITTEN_VAL[player])
      return (_GRADUATE_OFFSET + kittens).tolist()
    empty = np.flatnonzero(flat == _EMPTY)
    if hk > 0 and hc > 0:
      return np.concatenate([empty, _NUM_CELLS + empty]).tolist()
    if hk > 0:
      return empty.tolist()
    return (_NUM_CELLS + empty).tolist()

  def _apply_action(self, action):
    p = self._cur_player
    if action >= _GRADUATE_OFFSET:
      # Forced graduation: a kitten on the board becomes a cat in the pool.
      cell = action - _GRADUATE_OFFSET
      r, c = cell // _COLS, cell % _COLS
      self.board[r, c] = _EMPTY
      self._hand[p][1] += 1
      self._move_count += 1
      self._post_move(p)        # no piece placed → no boop
      return
    piece_type = action // _NUM_CELLS
    cell = action % _NUM_CELLS
    r, c = cell // _COLS, cell % _COLS
    self._hand[p][piece_type] -= 1
    self.board[r, c] = _PIECE_VALS[p][piece_type]
    self._boop(r, c, is_cat=(piece_type == 1))
    self._move_count += 1
    self._post_move(p)

  def _post_move(self, p):
    """Shared post-move resolution: win checks, promotion, turn handoff."""
    if self._move_count >= _MAX_GAME_LENGTH:
      self._is_terminal = True
      return
    for player in (p, 1 - p):
      if self._check_win(player):
        self._is_terminal = True
        self._winner = player
        return
    self._promote_kittens(p)
    self._promote_kittens(1 - p)
    for player in (p, 1 - p):
      if self._check_win(player):
        self._is_terminal = True
        self._winner = player
        return
    self._cur_player = 1 - p
    # With forced graduation a player is never permanently stuck: an empty pool
    # forces a graduation, and a board of eight cats is already a win. The guard
    # below is defensive only and should not trigger in normal play.
    if not self._legal_actions(self._cur_player):
      self._is_terminal = True

  def _action_to_string(self, player, action):
    if action >= _GRADUATE_OFFSET:
      cell = action - _GRADUATE_OFFSET
      r, c = cell // _COLS, cell % _COLS
      return f'p{player}:graduate@({r},{c})'
    pt = action // _NUM_CELLS
    cell = action % _NUM_CELLS
    r, c = cell // _COLS, cell % _COLS
    piece = 'cat' if pt else 'kitten'
    return f'p{player}:{piece}@({r},{c})'

  def is_terminal(self):
    return self._is_terminal

  def returns(self):
    if self._winner == 0:
      return [1.0, -1.0]
    if self._winner == 1:
      return [-1.0, 1.0]
    return [0.0, 0.0]

  def __str__(self):
    syms = {
        _EMPTY: '.', _P0_KITTEN: 'k', _P0_CAT: 'K',
        _P1_KITTEN: 'o', _P1_CAT: 'O',
    }
    rows = [
        ''.join(syms[self.board[r, c]] for c in range(_COLS))
        for r in range(_ROWS)
    ]
    rows.append(
        f'P0: {self._hand[0][0]}k {self._hand[0][1]}K  '
        f'P1: {self._hand[1][0]}k {self._hand[1][1]}K  '
        f'move={self._move_count}')
    return '\n'.join(rows)

  def _boop(self, r, c, is_cat):
    board = self.board
    for dr in (-1, 0, 1):
      for dc in (-1, 0, 1):
        if dr == 0 and dc == 0:
          continue
        nr, nc = r + dr, c + dc
        if not (0 <= nr < _ROWS and 0 <= nc < _COLS):
          continue
        neighbor = board[nr, nc]
        if neighbor == _EMPTY:
          continue
        neighbor_is_cat = neighbor == _P0_CAT or neighbor == _P1_CAT
        if not is_cat and neighbor_is_cat:
          continue
        dest_r, dest_c = nr + dr, nc + dc
        owner = 0 if (neighbor == _P0_KITTEN or neighbor == _P0_CAT) else 1
        n_type = 1 if neighbor_is_cat else 0
        if not (0 <= dest_r < _ROWS and 0 <= dest_c < _COLS):
          board[nr, nc] = _EMPTY
          self._hand[owner][n_type] += 1
        elif board[dest_r, dest_c] == _EMPTY:
          board[dest_r, dest_c] = neighbor
          board[nr, nc] = _EMPTY

  def _promote_kittens(self, player):
    # Faithful rule (rulebook p.4): a line of 3 of the player's own pieces —
    # kittens AND/OR cats mixed — graduates. Every kitten in the line becomes
    # a cat; every cat in the line simply returns to the pool; either way all
    # three board cells clear and the pool gains 3 cats (pieces conserved).
    # A pure 3-cats-in-a-row is a WIN, checked before this runs, so it never
    # reaches here as a live case.
    kv, cv = _KITTEN_VAL[player], _CAT_VAL[player]
    flat = self.board.reshape(-1)
    while True:
      mine = (flat == kv) | (flat == cv)
      if int(mine.sum()) < 3:
        return
      full = mine[_LINE_IDX].all(axis=1)
      if not full.any():
        return
      # Resolve ONE qualifying line per pass, chosen UNIFORMLY AT RANDOM
      # among all lines that currently qualify (not the player's choice —
      # that would need a new action type, which isn't worth the added
      # action-space complexity). Clearing the chosen line's cells
      # invalidates any OVERLAPPING line, so it is not picked again this
      # call — matching "choose one, leave the rest" for a connected run of
      # 4+ (fig.4). An independent (non-overlapping) line elsewhere still
      # qualifies and is resolved on the loop's next pass.
      candidates = _LINE_IDX[full]
      line = candidates[np.random.randint(len(candidates))]
      flat[line] = _EMPTY
      self._hand[player][1] += 3

  def _check_win(self, player):
    flat = self.board.reshape(-1)
    cats = flat == _CAT_VAL[player]
    n = int(cats.sum())
    if n >= _NUM_PIECES:        # win condition 1: all eight pieces are cats
      return True
    if n < 3:
      return False
    # Win condition 2: three cats in a row (orthogonal or diagonal).
    return bool(cats[_LINE_IDX].all(axis=1).any())


class BoopObserver:
  def __init__(self, params):
    if params:
      raise ValueError(f'Observation parameters not supported; passed {params}')
    board_size = 5 * _ROWS * _COLS
    self.tensor = np.zeros(board_size + 4, np.float32)
    self.dict = {
        'observation': np.reshape(self.tensor[:board_size], (5, _ROWS, _COLS)),
        'hand': self.tensor[board_size:],
    }

  def set_from(self, state, player):
    self.tensor.fill(0)
    obs = self.dict['observation']
    hand = self.dict['hand']
    b = state.board
    opp = 1 - player
    obs[0][b == _EMPTY] = 1.0
    obs[1][b == _KITTEN_VAL[player]] = 1.0
    obs[2][b == _CAT_VAL[player]] = 1.0
    obs[3][b == _KITTEN_VAL[opp]] = 1.0
    obs[4][b == _CAT_VAL[opp]] = 1.0
    hand[0] = state._hand[player][0] / _MAX_KITTENS
    hand[1] = state._hand[player][1] / _MAX_CATS
    hand[2] = state._hand[opp][0] / _MAX_KITTENS
    hand[3] = state._hand[opp][1] / _MAX_CATS

  def string_from(self, state, player):
    del player
    return str(state)


try:
    pyspiel.register_game(_GAME_TYPE, BoopGame)
except Exception:
    pass  # already registered in this process


# ═══════════════════════════════════════════════════════════════════════════════
# Thompson-MCTS tree ops — standalone copies of the notebook's semantics
# (Beta posteriors per edge, Thompson-sampled selection, count backups,
# loss-component virtual loss, MCTS-Solver). Workers are pure CPU: no torch
# import; every NN evaluation is shipped to the main process's GPU inference
# server.
# ═══════════════════════════════════════════════════════════════════════════════
import os

_WIN, _LOSS = 0, 1
_TERM_DRAW  = 2                        # move-cap terminal marker only
_W_VEC = np.array([1.0, 0.0])
_L_VEC = np.array([0.0, 1.0])
_D_VEC = np.array([0.5, 0.5])
_TERM_VECS = (_W_VEC, _L_VEC, _D_VEC)
_FLIP_TERM = np.array([_LOSS, _WIN, _TERM_DRAW], dtype=np.int8)   # one ply up
ALPHA_FLOOR = 0.05


class _TNode:
    __slots__ = ('player', 'legal', 'alphas', 'vloss', 'term', 'children')

    def __init__(self, player, legal, pwin_row, conf_row):
        self.player   = player
        self.legal    = np.asarray(legal, dtype=np.int32)
        pw = pwin_row[self.legal].astype(np.float64)
        cf = conf_row[self.legal].astype(np.float64)
        self.alphas   = np.maximum(np.stack([cf * pw, cf * (1.0 - pw)], axis=1),
                                   ALPHA_FLOOR)
        self.vloss    = np.zeros(len(legal), dtype=np.int32)
        self.term     = np.full(len(legal), -1, dtype=np.int8)
        self.children = [None] * len(legal)


def _thompson_values(node, rng, use_vloss):
    a = node.alphas
    if use_vloss and node.vloss.any():
        a = a.copy()
        a[:, _LOSS] += node.vloss
    g = rng.standard_gamma(a)
    p = g / (g.sum(axis=1, keepdims=True) + 1e-12)
    v = p[:, _WIN] - p[:, _LOSS]
    t = node.term
    v[t == _WIN]      =  2.0
    v[t == _LOSS]     = -2.0
    v[t == _TERM_DRAW] = 0.0
    return v, p


def _mean_values(node):
    a  = node.alphas
    a0 = a.sum(axis=1)
    v  = (a[:, _WIN] - a[:, _LOSS]) / a0
    t  = node.term
    v[t == _WIN]      =  2.0
    v[t == _LOSS]     = -2.0
    v[t == _TERM_DRAW] = 0.0
    return v


def _select_leaf(root, root_state, rng):
    node  = root
    state = root_state.clone()
    path  = []
    while True:
        v, _ = _thompson_values(node, rng, use_vloss=True)
        idx  = int(v.argmax())
        node.vloss[idx] += 1
        path.append((node, idx))
        if node.term[idx] >= 0:
            return path, None, None
        state.apply_action(int(node.legal[idx]))
        if state.is_terminal():
            r = state.returns()[node.player]
            node.term[idx] = _WIN if r > 0 else (_LOSS if r < 0 else _TERM_DRAW)
            return path, None, None
        child = node.children[idx]
        if child is None:
            return path, state, (node, idx)
        node = child


def _leaf_outcome(leaf, rng):
    v, p = _thompson_values(leaf, rng, use_vloss=False)
    return _W_VEC if rng.random_sample() < p[int(v.argmax()), _WIN] else _L_VEC


def _backup(path, wvec, outcome_player):
    flipped = wvec[::-1]
    for node, idx in path:
        node.vloss[idx] -= 1
        node.alphas[idx] += wvec if node.player == outcome_player else flipped


def _node_solved_outcome(node):
    """Proven outcome for node.player if solved, else None (MCTS-Solver): WIN
    as soon as any edge is a proven win; LOSS/DRAW only once all are proven."""
    t = node.term
    if (t == _WIN).any():
        return _WIN
    if (t >= 0).all():
        return _TERM_DRAW if (t == _TERM_DRAW).any() else _LOSS
    return None


def _propagate_solved(path):
    for k in range(len(path) - 1, 0, -1):
        out = _node_solved_outcome(path[k][0])
        if out is None:
            break
        parent, pidx = path[k - 1]
        if parent.term[pidx] >= 0:
            break
        parent.term[pidx] = _FLIP_TERM[out]


def _backup_terminal(path):
    node, idx = path[-1]
    _backup(path, _TERM_VECS[int(node.term[idx])], node.player)
    _propagate_solved(path)


def root_target(root, conf_cap):
    a = root.alphas.copy()
    t = root.term
    if (t >= 0).any():
        a[t == _WIN]       = (conf_cap, ALPHA_FLOOR)
        a[t == _LOSS]      = (ALPHA_FLOOR, conf_cap)
        a[t == _TERM_DRAW] = (0.5 * conf_cap, 0.5 * conf_cap)
    s = a.sum(axis=1)
    a *= np.minimum(1.0, conf_cap / np.maximum(s, 1e-9))[:, None]
    tgt = np.zeros((_NUM_ACTIONS, 2), dtype=np.float32)
    tgt[root.legal] = a
    return tgt


def root_pick(root, rng, thompson):
    if thompson:
        v, _ = _thompson_values(root, rng, use_vloss=False)
    else:
        v = _mean_values(root)
    return int(root.legal[int(v.argmax())])


def run_worker(worker_id, req_q, resp_q, pool_resp_q, episode_q, cfg):
    """Pipelined self-play worker (leapfrog halves, identical protocol to the
    KataZero worker): while one half's NN request is in flight on the central
    server, the CPU does tree work for the other half. NN requests are tagged
    'live' (the training net) or a benchmark label (a frozen opponent-diversity
    net); the server routes them and answers 'live' on resp_q, pool nets on
    pool_resp_q."""
    rng  = np.random.RandomState(cfg['seed'] + worker_id * 7919)
    game = pyspiel.load_game('python_boop')
    conf_cap  = cfg['conf_cap']
    pool_prob = cfg.get('pool_prob', 0.0)
    pool_dir  = cfg.get('checkpoint_dir')

    def new_game():
        sims = (cfg['fast_sims'] if rng.rand() < cfg['fast_prob']
                else cfg['full_sims'])
        slot = {'state': game.new_initial_state(), 'hist': [],
                'move': 0, 'sims': sims, 'root': None, 'n': 0, 'pool': None}
        # Opponent diversity: some games have one side played by a FROZEN
        # historical checkpoint instead of the live net mirror-matching itself.
        # Only the live side's moves feed the replay buffer.
        if pool_prob > 0 and pool_dir:
            try:
                labels = [f[6:-3] for f in os.listdir(pool_dir)
                          if f.startswith('bench_') and f.endswith('.pt')]
            except OSError:
                labels = []
            if labels and rng.rand() < pool_prob:
                slot['pool'] = {'label': labels[rng.randint(len(labels))],
                                'side': int(rng.randint(0, 2))}
        return slot

    slots = [new_game() for _ in range(cfg['games_per_worker'])]
    mid = max(1, cfg['games_per_worker'] // 2)
    halves = [list(range(mid)), list(range(mid, cfg['games_per_worker']))]

    def collect(idxs):
        """Gather root expansions + one leaf wave per game. Terminal edges are
        backed up immediately; pool-opponent turns and solver-proven roots are
        skipped. Returns (evals, paths, obs): `evals` are the unique NN jobs
        (obs rows aligned with them), `paths` every pending leaf awaiting an
        outcome."""
        evals, paths, obs = [], [], []
        seen = set()
        for i in idxs:
            s = slots[i]
            st0 = s['state']
            pool = s['pool']
            if pool is not None and st0.current_player() == pool['side']:
                continue         # next move belongs to the frozen opponent
            if s['root'] is None:
                evals.append(('root', i, None, None))
                obs.append(np.asarray(
                    st0.observation_tensor(st0.current_player()),
                    dtype=np.float32))
                continue
            if _node_solved_outcome(s['root']) is not None:
                continue         # MCTS-Solver: root proven — finalized below
            wave = min(cfg['wave'], s['sims'] - s['n'])
            for _ in range(max(wave, 0)):
                path, st, edge = _select_leaf(s['root'], st0, rng)
                if st is None:
                    _backup_terminal(path)
                    s['n'] += 1
                    continue
                node, idx = edge
                paths.append((i, path, node, idx))
                if (id(node), idx) not in seen:
                    seen.add((id(node), idx))
                    evals.append(('leaf', node, idx,
                                  (st.current_player(), st.legal_actions())))
                    obs.append(np.asarray(
                        st.observation_tensor(st.current_player()),
                        dtype=np.float32))
        return evals, paths, obs

    def apply_and_advance(idxs, evals, paths, resp):
        """Expand a completed wave (roots + leaves), draw and back up leaf
        outcomes, then finish any completed (or solver-proven) searches: record
        the posterior target and play the Thompson-sampled move."""
        if evals:
            p_win, conf = resp
            for e, p_row, c_row in zip(evals, p_win, conf):
                if e[0] == 'root':
                    i = e[1]
                    st = slots[i]['state']
                    slots[i]['root'] = _TNode(st.current_player(),
                                              st.legal_actions(), p_row, c_row)
                else:
                    _, node, idx, (player, legal) = e
                    node.children[idx] = _TNode(player, legal, p_row, c_row)
        for i, path, node, idx in paths:
            leaf = node.children[idx]
            _backup(path, _leaf_outcome(leaf, rng), leaf.player)
            slots[i]['n'] += 1
        for i in idxs:
            s = slots[i]
            if s['root'] is None:
                continue
            if s['n'] < s['sims'] and _node_solved_outcome(s['root']) is None:
                continue
            state = s['state']
            o = np.asarray(state.observation_tensor(state.current_player()),
                           dtype=np.float32)
            s['hist'].append((o, root_target(s['root'], conf_cap)))
            action = root_pick(s['root'], rng,
                               thompson=(s['move'] < cfg['temp_threshold']))
            state.apply_action(action)
            s['move'] += 1
            s['root'] = None
            s['n'] = 0
            if state.is_terminal():
                episode_q.put(s['hist'])
                slots[i] = new_game()

    def resolve_pool_moves(idxs):
        """For any game whose CURRENT mover is the frozen-opponent seat, resolve
        that single move with ONE direct request to the pool net (tagged by its
        benchmark label) and a greedy p_win argmax — no search tree, since these
        moves are never training targets. Turns strictly alternate, so at most
        one such move is pending per game here."""
        for i in idxs:
            s = slots[i]
            pool = s['pool']
            if pool is None:
                continue
            state = s['state']
            if state.current_player() != pool['side']:
                continue
            obs = np.asarray(state.observation_tensor(state.current_player()),
                             dtype=np.float32)
            req_q.put((worker_id, pool['label'], obs[None]))
            p_win, _conf = pool_resp_q.get()
            legal = state.legal_actions()
            action = legal[int(np.argmax(p_win[0][legal]))]
            state.apply_action(action)
            s['move'] += 1
            if state.is_terminal():
                episode_q.put(s['hist'])
                slots[i] = new_game()

    # Leapfrog loop. Send/receive alternate A,B,A,B so the per-worker FIFO
    # response queue always delivers the response for the half we expect;
    # the `sent` flag keeps alignment when a half had nothing to evaluate.
    inflight = [None, None]          # per half: (evals, paths, sent)
    while True:
        for h in (0, 1):
            if inflight[h] is not None:
                evals, paths, sent = inflight[h]
                resp = resp_q.get() if sent else None
                apply_and_advance(halves[h], evals, paths, resp)
                inflight[h] = None
            resolve_pool_moves(halves[h])
            evals, paths, obs = collect(halves[h])
            sent = False
            if evals:
                req_q.put((worker_id, 'live', np.stack(obs)))
                sent = True
            inflight[h] = (evals, paths, sent)
'''

pathlib.Path('boop_thompson_worker.py').write_text(_WORKER_SRC, encoding='utf-8')

import multiprocessing as _mp
import threading
import queue as _queue
import time as _time
import os as _os
import sys as _sys


class MPSelfPlayPool:
    """n_workers CPU processes play games; an inference-server thread in this
    process batches ALL their NN requests into single forward passes on
    `device`. Exposes the same .episodes() generator as ThompsonParallelSelfPlay.
    `lock` serializes model access between the server thread and training."""

    def __init__(self, network, device, n_workers, cfg, batch_window_s=0.002,
                 checkpoint_dir=None, channels=None, num_blocks=None):
        self.network = network
        self.device = device
        self.checkpoint_dir = checkpoint_dir
        self.channels = channels
        self.num_blocks = num_blocks
        self._pool_nets = {}          # benchmark label -> frozen CPU ThompsonNet
        self.lock = threading.Lock()
        self._stop = threading.Event()
        ctx = _mp.get_context('spawn')            # Windows-compatible
        self.req_q = ctx.Queue()
        self.episode_q = ctx.Queue(maxsize=64)    # backpressure on workers
        self.resp_qs = [ctx.Queue() for _ in range(n_workers)]
        self.pool_resp_qs = [ctx.Queue() for _ in range(n_workers)]
        self.window = batch_window_s
        # Initialize the autograd engine's per-device state from the MAIN
        # thread before any other thread touches the device: otherwise the
        # first backward races the server thread's device use and trips
        # torch-directml's "device_ready_queues_" INTERNAL ASSERT.
        if str(device) != 'cpu':
            _t = torch.zeros(4, device=device, requires_grad=True)
            (_t * 2.0).sum().backward()
        if _os.getcwd() not in _sys.path:
            _sys.path.insert(0, _os.getcwd())
        import boop_thompson_worker
        self.procs = [ctx.Process(target=boop_thompson_worker.run_worker,
                                  args=(i, self.req_q, self.resp_qs[i],
                                        self.pool_resp_qs[i],
                                        self.episode_q, cfg), daemon=True)
                      for i in range(n_workers)]
        for p in self.procs:
            p.start()
        self.server = threading.Thread(target=self._serve, daemon=True)
        self.server.start()

    def _get_net(self, net_id):
        """'live' -> the training network on its real device (needs the lock:
        DirectML thread-safety + training mutates its weights concurrently).
        Any other net_id -> a frozen benchmark checkpoint, lazily loaded once
        and cached, always run on CPU (no lock: never mutated). Pool batches are
        tiny and infrequent, so CPU inference for them costs nothing worth
        optimizing."""
        if net_id == 'live':
            return self.network, self.device, True
        net = self._pool_nets.get(net_id)
        if net is None:
            path = _os.path.join(self.checkpoint_dir, f'bench_{net_id}.pt')
            net = ThompsonNet(self.channels, self.num_blocks)
            net.load_state_dict(torch.load(path, map_location='cpu',
                                           weights_only=True))
            net.eval()
            self._pool_nets[net_id] = net
        return net, 'cpu', False

    def _serve(self):
        while not self._stop.is_set():
            try:
                reqs = [self.req_q.get(timeout=0.1)]
            except _queue.Empty:
                continue
            # Short batching window: gather concurrent workers' requests so
            # the GPU sees one big batch instead of several small ones.
            deadline = _time.monotonic() + self.window
            while _time.monotonic() < deadline and len(reqs) < 64:
                try:
                    reqs.append(self.req_q.get_nowait())
                except _queue.Empty:
                    _time.sleep(0.0003)
            # Group by net_id: almost always all 'live', occasionally a request
            # or two tagged with a frozen benchmark label (opponent-diversity
            # games) mixed in.
            groups = {}
            for wid, net_id, obs in reqs:
                groups.setdefault(net_id, []).append((wid, obs))
            for net_id, group in groups.items():
                net, net_device, needs_lock = self._get_net(net_id)
                obs = np.concatenate([o for _, o in group], axis=0)
                xin = _obs_to_9ch(obs)
                # EVERY device op on the LIVE net under the lock (upload,
                # forward, download): DirectML is not safe under concurrent
                # access from two threads. Frozen CPU pool nets need no lock.
                if needs_lock:
                    with self.lock, torch.no_grad():
                        x = torch.from_numpy(xin).to(net_device)
                        p_win, conf = net(x)
                        pr = p_win.cpu().numpy()
                        cf = conf.cpu().numpy()
                else:
                    with torch.no_grad():
                        x = torch.from_numpy(xin).to(net_device)
                        p_win, conf = net(x)
                        pr = p_win.cpu().numpy()
                        cf = conf.cpu().numpy()
                off = 0
                target_qs = self.resp_qs if net_id == 'live' else self.pool_resp_qs
                for wid, o in group:
                    n = o.shape[0]
                    target_qs[wid].put((pr[off:off + n], cf[off:off + n]))
                    off += n

    def episodes(self):
        while True:
            yield self.episode_q.get()

    def shutdown(self):
        self._stop.set()
        try:
            self.server.join(timeout=2.0)
        except Exception:
            pass
        for p in self.procs:
            p.terminate()
        for p in self.procs:
            p.join(timeout=2.0)
        for q in ([self.req_q, self.episode_q] + self.resp_qs
                  + self.pool_resp_qs):
            try:
                q.close(); q.cancel_join_thread()
            except Exception:
                pass

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
NUM_EPISODES     = 20_000  # run budget only — the LR schedule no longer
                           # depends on it, so extend freely
FULL_MCTS_SIMS   = 1000    # high-quality self-play (25% of games)
FAST_MCTS_SIMS   = 250     # fast self-play (75%) — more sims = sharper posteriors
FAST_GAME_PROB   = 0.75    # fraction of episodes using FAST_MCTS_SIMS
# ── Hardware parallelism (AMD GPU via DirectML + 6-core CPU) ─────────────────
USE_WORKERS      = True    # multiprocessing self-play with a central GPU server
SELFPLAY_WORKERS = 4       # CPU game/tree processes
GAMES_PER_WORKER = 16      # per worker, split into two pipelined halves of 8
WORKER_WAVE      = 8       # leaves/game/wave; server batches all workers' requests
N_PARALLEL_GAMES = 8       # (single-process fallback when USE_WORKERS=False)
WAVE_PER_GAME    = 8       # MCTS leaves per game per wave
TEMP_THRESHOLD   = 20      # Thompson-sampled moves per game; posterior-mean
                           # argmax afterwards (the annealing analogue)
CONF_CAP         = 100.0   # posterior targets rescaled (mean-preserving) to at
                           # most this total count — caps certainty growth
                           # across generations (CONF_MAX clamps the net's side)
SELFPLAY_POOL_PROB = 0.15  # fraction of self-play games where one random side
                           # is played by a FROZEN benchmark instead of the live
                           # net mirror-matching itself — counters the
                           # non-transitive "strategy cycling" pure self-play can
                           # fall into (only the live side's moves are training
                           # targets). 0 disables it.
EVAL_EVERY       = 50      # QUICK eval: live net vs the most recent benchmark
DEEP_EVAL_EVERY  = 500     # DEEP eval: round-robin tournaments + new benchmark
QUICK_EVAL_GAMES = 20      # games vs the most recent benchmark (quick pulse)
TOURNEY_GAMES_PER_PAIR = 20  # round-robin games per pair (10 each colour)
TOURNEY_FULL_RR  = False   # linear-cost tournaments (new model vs everyone +
                           # REFRESH_PAIRS random old pairs)
REFRESH_PAIRS    = 4
# Independent evaluation tracks: (name, MCTS simulations per move). Each track
# keeps its OWN Elo table and W/D/L matrix. sims=0 is the raw network.
EVAL_TRACKS      = [('policy', 0), ('mcts25', 25), ('mcts50', 50)]
EVAL_OPENING_PLIES = 4     # random opening half-moves per eval game (variety)
START_ELO        = 1000.0  # random and the initial net seed here
K_FACTOR         = 32.0    # Elo update step
BATCH_SIZE       = 512
TRAIN_STEPS_PER_EP = 8     # gradient steps per self-play game
MAX_BUFFER       = 400_000
CHANNELS         = 128
NUM_BLOCKS       = 8
LR_PEAK          = 2e-3
LR_WARMUP_EPS    = 100      # linear warmup episodes
LR_DECAY_EPS     = 12_000   # cosine horizon — decoupled from NUM_EPISODES
WEIGHT_DECAY     = 1e-4
LR_MIN_FACTOR    = 0.10     # cosine decay floor (fraction of LR_PEAK)
CHECKPOINT_DIR   = 'boop_checkpoints_thompson'  # NOT shared with the KataZero
                           # runs — different net, different state dicts
RESUME_FROM_CHECKPOINT = True  # auto-resume from CHECKPOINT_DIR/latest.pt

# ── Setup ─────────────────────────────────────────────────────────────────────
game         = pyspiel.load_game('python_boop')
base_network = ThompsonNet(CHANNELS, NUM_BLOCKS).to(DEVICE)  # uncompiled; shares
network      = base_network                                  # weights w/ compiled

# torch.compile: same policy as the KataZero notebook — skipped on DirectML and
# whenever Inductor has no C++ backend; falls back to eager on lazy failure.
network = base_network
if _BACKEND != 'directml' and hasattr(torch, 'compile'):
    import shutil, platform, logging
    _cc = (shutil.which('cl') if platform.system() == 'Windows'
           else shutil.which('g++') or shutil.which('gcc') or shutil.which('clang'))
    if _cc is None:
        print('torch.compile: skipped (no C++ compiler for Inductor) — eager mode')
    else:
        logging.getLogger('torch.fx.experimental.symbolic_shapes').setLevel(logging.ERROR)
        try:
            _compiled = torch.compile(base_network, dynamic=True)
            with torch.no_grad():                   # force compilation now
                _compiled(torch.zeros(1, 9, 6, 6, device=DEVICE))
            network = _compiled
            print('torch.compile: enabled')
        except Exception as _e:
            print(f'torch.compile: disabled ({type(_e).__name__}) — running eager mode')

class LerpFreeAdamW(torch.optim.Optimizer):
    """AdamW built from mul_/add_/addcmul_/addcdiv_ only — DirectML lacks
    aten::lerp, which torch's AdamW uses in both its foreach and single-tensor
    paths (each step would round-trip every parameter through the CPU).
    Identical math to torch.optim.AdamW (decoupled decay, bias correction)."""

    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8,
                 weight_decay=1e-2):
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps,
                                      weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr, (b1, b2) = group['lr'], group['betas']
            eps, wd = group['eps'], group['weight_decay']
            for p in group['params']:
                if p.grad is None:
                    continue
                st = self.state[p]
                if not st:
                    st['step'] = 0
                    st['exp_avg'] = torch.zeros_like(p)
                    st['exp_avg_sq'] = torch.zeros_like(p)
                st['step'] += 1
                t = int(st['step'])
                m, v = st['exp_avg'], st['exp_avg_sq']
                p.mul_(1.0 - lr * wd)                     # decoupled decay
                m.mul_(b1).add_(p.grad, alpha=1.0 - b1)   # lerp-free
                v.mul_(b2).addcmul_(p.grad, p.grad, value=1.0 - b2)
                bc1 = 1.0 - b1 ** t
                bc2 = 1.0 - b2 ** t
                denom = (v.sqrt() / (bc2 ** 0.5)).add_(eps)
                p.addcdiv_(m, denom, value=-(lr / bc1))
        return loss


if _BACKEND == 'directml':
    optimizer = LerpFreeAdamW(network.parameters(),
                              lr=LR_PEAK, weight_decay=WEIGHT_DECAY)
else:
    optimizer = torch.optim.AdamW(network.parameters(),
                                  lr=LR_PEAK, weight_decay=WEIGHT_DECAY)

def _lr_lambda(ep):
    if ep < LR_WARMUP_EPS:
        return ep / max(LR_WARMUP_EPS, 1)
    frac = min(1.0, (ep - LR_WARMUP_EPS) / max(LR_DECAY_EPS - LR_WARMUP_EPS, 1))
    # float(): np.cos returns a numpy scalar, which otherwise lands in the
    # scheduler state and trips torch.load(weights_only=True) on resume.
    return float(max(LR_MIN_FACTOR, 0.5 * (1.0 + np.cos(np.pi * frac))))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, _lr_lambda)


# ── Loss: KL( Beta(MCTS posterior) ‖ Beta(predicted prior) ) ──────────────────
# Closed form (a Beta is a 2-component Dirichlet) with the target first
# (forward KL), averaged over LEGAL actions (zero target rows = illegal).

def beta_kl_loss(p_win, conf, target):
    """p_win, conf (B,108) → predicted prior β = conf·(p_win, 1−p_win).
    target (B,108,2): posterior (win, loss) counts; zero rows are masked out.
    Returns (mean KL over legal actions, legal mask). Uses lgamma/digamma —
    pure math, run wherever its input tensors live (GPU on CUDA, CPU on
    DirectML; see kl_loss_backward)."""
    mask = (target.sum(dim=-1) > 0).float()                    # (B, 108)
    t    = target.clamp_min(ALPHA_FLOOR)
    b    = (conf.unsqueeze(-1)
            * torch.stack([p_win, 1.0 - p_win], dim=-1)).clamp_min(ALPHA_FLOOR)
    t0   = t.sum(dim=-1)
    b0   = b.sum(dim=-1)
    kl = (torch.lgamma(t0) - torch.lgamma(t).sum(dim=-1)
          - torch.lgamma(b0) + torch.lgamma(b).sum(dim=-1)
          + ((t - b) * (torch.digamma(t)
                        - torch.digamma(t0).unsqueeze(-1))).sum(dim=-1))
    return (kl * mask).sum() / mask.sum().clamp_min(1.0), mask


# DirectML has no lgamma/digamma kernels (as with lerp / native_group_norm
# elsewhere in this notebook). Rather than gamble on a CPU fallback or on
# cross-device autograd, on that backend we SPLIT the backward at the device
# boundary: the tiny (B×108) head outputs are copied to CPU, the KL loss and
# its gradient are computed there, and those gradients are handed back to the
# GPU graph as seed gradients. No lgamma runs on the GPU, and the only
# device transfers are plain tensor copies (exactly what every forward already
# does to upload inputs) — never an autograd edge spanning two devices.
_KL_SPLIT = (_BACKEND == 'directml')

def kl_loss_backward(p_win, conf, target):
    """Compute the Beta-KL loss and run its backward into the network params.
    Returns the scalar loss value. `optimizer.zero_grad()` must precede this;
    `clip_grad_norm_` + `optimizer.step()` follow it."""
    if _KL_SPLIT:
        p_cpu = p_win.detach().cpu().requires_grad_(True)
        c_cpu = conf.detach().cpu().requires_grad_(True)
        loss_cpu, _ = beta_kl_loss(p_cpu, c_cpu, target.detach().cpu())
        loss_cpu.backward()
        gp = p_cpu.grad if p_cpu.grad is not None else torch.zeros_like(p_cpu)
        gc = c_cpu.grad if c_cpu.grad is not None else torch.zeros_like(c_cpu)
        torch.autograd.backward([p_win, conf],
                                [gp.to(p_win.device), gc.to(conf.device)])
        return float(loss_cpu.item())
    loss, _ = beta_kl_loss(p_win, conf, target)
    loss.backward()
    return float(loss.item())


# Self-play source: worker pool (multiprocess + GPU server) or in-process.
import os
import threading
try:
    self_play.shutdown()               # re-running this cell: stop old workers
except (NameError, AttributeError):
    pass
if USE_WORKERS:
    _wcfg = dict(seed=int(np.random.randint(1_000_000)),
                 games_per_worker=GAMES_PER_WORKER, wave=WORKER_WAVE,
                 fast_sims=FAST_MCTS_SIMS, full_sims=FULL_MCTS_SIMS,
                 fast_prob=FAST_GAME_PROB, temp_threshold=TEMP_THRESHOLD,
                 conf_cap=CONF_CAP, pool_prob=SELFPLAY_POOL_PROB,
                 checkpoint_dir=CHECKPOINT_DIR)
    self_play = MPSelfPlayPool(network, DEVICE, SELFPLAY_WORKERS, _wcfg,
                               checkpoint_dir=CHECKPOINT_DIR,
                               channels=CHANNELS, num_blocks=NUM_BLOCKS)
    torch.set_num_threads(max(2, (os.cpu_count() or 6) - SELFPLAY_WORKERS))
else:
    self_play = ThompsonParallelSelfPlay(game, network, DEVICE,
                                         n_parallel=N_PARALLEL_GAMES,
                                         wave_per_game=WAVE_PER_GAME,
                                         fast_sims=FAST_MCTS_SIMS,
                                         full_sims=FULL_MCTS_SIMS,
                                         fast_prob=FAST_GAME_PROB,
                                         temp_threshold=TEMP_THRESHOLD,
                                         conf_cap=CONF_CAP,
                                         pool_prob=SELFPLAY_POOL_PROB,
                                         checkpoint_dir=CHECKPOINT_DIR,
                                         channels=CHANNELS,
                                         num_blocks=NUM_BLOCKS)
episode_stream = self_play.episodes()
# Serializes model access between the inference-server thread and training.
MODEL_LOCK = getattr(self_play, 'lock', None) or threading.Lock()

# Small-batch inference (eval games, tournaments) runs on CPU replicas — batch-1
# forwards on DirectML are latency-bound and would dominate eval time.
EVAL_DEVICE = 'cpu'

def _cpu_sd(net):
    return {k: v.detach().cpu() for k, v in net.state_dict().items()}

def _cpu_clone(net):
    m = ThompsonNet(CHANNELS, NUM_BLOCKS)
    m.load_state_dict(_cpu_sd(net))
    m.eval()
    return m

def _to_cpu_tree(obj):
    if torch.is_tensor(obj):
        return obj.detach().cpu()
    if isinstance(obj, dict):
        return {k: _to_cpu_tree(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_to_cpu_tree(v) for v in obj]
    return obj

replay_buffer = []
# Tournament pool. `random` and the untrained `0` net both start at START_ELO.
_init_snap = _cpu_clone(base_network)
eval_net   = _cpu_clone(base_network)  # refreshed at every eval
benchmarks = [{'label': 'random', 'net': None},
              {'label': '0',      'net': _init_snap}]
# Per-track Elo tables and head-to-head matrices (independent between tracks).
elos = {name: {'random': START_ELO, '0': START_ELO} for name, _ in EVAL_TRACKS}
wdl  = {name: defaultdict(lambda: [0, 0, 0])         for name, _ in EVAL_TRACKS}
hist = {'ep': [], 'kl': [], 'v_err': [], 'conf_t': [], 'conf_p': [],
        'quick_ep': [], 'q_w': [], 'q_d': [], 'q_l': [],
        'deep_ep': [], 'elo_snap': []}   # elo_snap: list of {track: {label: elo}}

# ── Checkpointing: survive crashes, resume indefinitely ──────────────────────
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
_LATEST_CKPT = os.path.join(CHECKPOINT_DIR, 'latest.pt')

def _save_benchmark_net(label, net):
    if net is not None:
        torch.save(net.state_dict(),
                   os.path.join(CHECKPOINT_DIR, f'bench_{label}.pt'))

def save_checkpoint(ep):
    # Copy state to CPU first: DirectML (privateuseone) tensors may not pickle.
    with MODEL_LOCK:
        _model_sd = _to_cpu_tree(base_network.state_dict())
        _opt_sd   = _to_cpu_tree(optimizer.state_dict())
    torch.save({'ep': ep,
                'model': _model_sd,
                'optimizer': _opt_sd,
                'scheduler': scheduler.state_dict(),
                'elos': elos,
                'wdl': {name: dict(w) for name, w in wdl.items()},
                'hist': hist,
                'bench_labels': [b['label'] for b in benchmarks]}, _LATEST_CKPT)

start_ep = 1
if RESUME_FROM_CHECKPOINT and os.path.exists(_LATEST_CKPT):
    try:
        _ck = torch.load(_LATEST_CKPT, map_location='cpu', weights_only=True)
    except Exception:                     # older ckpt with a numpy scalar inside
        _ck = torch.load(_LATEST_CKPT, map_location='cpu', weights_only=False)
    base_network.load_state_dict(_ck['model'])
    optimizer.load_state_dict(_ck['optimizer'])
    scheduler.load_state_dict(_ck['scheduler'])
    elos = _ck['elos']
    wdl  = {name: defaultdict(lambda: [0, 0, 0], w)
            for name, w in _ck['wdl'].items()}
    hist = _ck['hist']
    benchmarks = [{'label': 'random', 'net': None}]
    for _lbl in _ck['bench_labels']:
        if _lbl == 'random':
            continue
        _bn = ThompsonNet(CHANNELS, NUM_BLOCKS)      # benchmarks live on CPU
        _bn.load_state_dict(torch.load(
            os.path.join(CHECKPOINT_DIR, f'bench_{_lbl}.pt'),
            map_location='cpu', weights_only=True))
        _bn.eval()
        benchmarks.append({'label': _lbl, 'net': _bn})
    start_ep = _ck['ep'] + 1
    print(f'Resumed from checkpoint: episode {_ck["ep"]}, pool {len(benchmarks)}')


def _run_all_tracks(focus=None):
    for name, sims in EVAL_TRACKS:
        run_tournament(benchmarks, elos[name], wdl[name], game, EVAL_DEVICE,
                       TOURNEY_GAMES_PER_PAIR, K_FACTOR, sims,
                       opening_plies=EVAL_OPENING_PLIES,
                       focus_label=focus, refresh_pairs=REFRESH_PAIRS)
    hist['deep_ep'].append(ep_now[0])
    hist['elo_snap'].append({name: dict(elos[name]) for name, _ in EVAL_TRACKS})


def _print_tracks(prefix):
    for name, _ in EVAL_TRACKS:
        ladder = '  '.join(f'{b["label"]}={elos[name][b["label"]]:.0f}'
                           for b in sorted(benchmarks, key=lambda b: -elos[name][b['label']]))
        print(f'{prefix} {name:>7} Elos (strong→weak): {ladder}')


n_params = sum(p.numel() for p in network.parameters() if p.requires_grad)
print(f'Device: {DEVICE}  |  ThompsonNet params: {n_params:,}')
print(f'{NUM_EPISODES} eps | fast {FAST_MCTS_SIMS} sims ({FAST_GAME_PROB:.0%}) '
      f'/ full {FULL_MCTS_SIMS} sims | Thompson selection + MCTS-Solver, conf cap '
      f'{CONF_CAP:.0f} | pool {SELFPLAY_POOL_PROB:.0%} | 8x aug | '
      f'{TRAIN_STEPS_PER_EP} steps/ep')
print(f'Quick eval every {EVAL_EVERY} eps (vs latest benchmark); deep tournaments '
      f'every {DEEP_EVAL_EVERY} eps | tracks: {[f"{n}({s})" for n, s in EVAL_TRACKS]}')
print(f'Checkpoints: {CHECKPOINT_DIR} (resume={RESUME_FROM_CHECKPOINT})')
if USE_WORKERS:
    print(f'Self-play: {SELFPLAY_WORKERS} worker processes × {GAMES_PER_WORKER} games '
          f'(wave {WORKER_WAVE}) → GPU inference server on {DEVICE}')
print('-' * 72)

ep_now = [start_ep - 1]
if start_ep == 1:
    # Iteration-0 tournaments: random vs the untrained net, one per track.
    _save_benchmark_net('0', _init_snap)
    _run_all_tracks()
    _print_tracks('Iter     0 |')
    save_checkpoint(0)
print('-' * 72)

for ep in range(start_ep, NUM_EPISODES + 1):
    ep_now[0] = ep

    # 1. Pull the next finished self-play game (list of (obs, target) samples)
    network.eval()
    raw_data = next(episode_stream)

    # 2. Augment each move with all 8 board symmetries (8× free data)
    for obs, tgt in raw_data:
        replay_buffer.extend(augment_sample(obs, tgt))
    if len(replay_buffer) > MAX_BUFFER:
        del replay_buffer[:-MAX_BUFFER]

    # 3. Train: KL(posterior ‖ predicted prior) per legal action
    network.train()
    kls, v_errs, conf_ts, conf_ps = [], [], [], []
    if len(replay_buffer) >= BATCH_SIZE:
        for _ in range(TRAIN_STEPS_PER_EP):
            batch = random.sample(replay_buffer, BATCH_SIZE)
            # The ENTIRE step body sits under the lock: every device op must be
            # serialized against the inference-server thread (DirectML is not
            # thread-safe).
            with MODEL_LOCK:
                x_b   = batch_to_tensor([s[0] for s in batch], DEVICE)
                tgt_b = torch.from_numpy(
                            np.stack([s[1] for s in batch]).astype(np.float32)
                        ).to(DEVICE)

                p_win, conf = network(x_b)
                optimizer.zero_grad()
                loss_val = kl_loss_backward(p_win, conf, tgt_b)
                torch.nn.utils.clip_grad_norm_(network.parameters(), 1.0)
                optimizer.step()

                with torch.no_grad():
                    # Diagnostics only — computed on CPU copies (tiny tensors)
                    # so no exotic op ever touches the DirectML device. Masked
                    # means over legal actions via (x·mask).sum()/mask.sum(),
                    # not boolean indexing.
                    tg = tgt_b.detach().cpu()
                    pw = p_win.detach().cpu()
                    cf = conf.detach().cpu()
                    mf    = (tg.sum(dim=-1) > 0).float()      # (B, 108)
                    denom = mf.sum().clamp_min(1.0)
                    t     = tg.clamp_min(ALPHA_FLOOR)
                    t0    = t.sum(dim=-1)
                    tp    = t[..., 0] / t0                    # target mean p_win
                    v_errs.append((((tp - pw).abs() * mf).sum() / denom).item())
                    conf_ts.append(((t0 * mf).sum() / denom).item())
                    conf_ps.append(((cf * mf).sum() / denom).item())
                kls.append(loss_val)
        scheduler.step()

    if ep % EVAL_EVERY != 0:
        continue

    # 4. Evaluation (CPU replica: batch-1 DirectML inference is latency-bound)
    network.eval()
    with MODEL_LOCK:
        eval_net.load_state_dict(_cpu_sd(base_network))
    mkl = float(np.mean(kls))     if kls     else float('nan')
    mve = float(np.mean(v_errs))  if v_errs  else float('nan')
    mct = float(np.mean(conf_ts)) if conf_ts else float('nan')
    mcp = float(np.mean(conf_ps)) if conf_ps else float('nan')
    hist['ep'].append(ep); hist['kl'].append(mkl); hist['v_err'].append(mve)
    hist['conf_t'].append(mct); hist['conf_p'].append(mcp)
    lr_now = optimizer.param_groups[0]['lr']

    if ep % DEEP_EVAL_EVERY == 0:
        # DEEP: add current net as a new benchmark, warm-starting each track's
        # Elo from the previous checkpoint, then run one round-robin per track.
        new_label = str(ep)
        for name, _ in EVAL_TRACKS:
            elos[name][new_label] = elos[name][benchmarks[-1]['label']]
        with MODEL_LOCK:
            snap = _cpu_clone(base_network)
        benchmarks.append({'label': new_label, 'net': snap})
        _run_all_tracks(None if TOURNEY_FULL_RR else new_label)
        print(f'Ep {ep:5d} | KL={mkl:.3f} vErr={mve:.3f} a0 t/p={mct:.1f}/{mcp:.1f} '
              f'| DEEP tournaments (pool {len(benchmarks)}) | lr={lr_now:.2e}')
        _print_tracks('        |')
        _save_benchmark_net(new_label, snap)
        save_checkpoint(ep)
    else:
        # QUICK: search-free pulse vs the most recent benchmark (no Elo change).
        ref = benchmarks[-1]
        w, d, l = play_match(eval_net, ref['net'], game,
                             QUICK_EVAL_GAMES, EVAL_DEVICE)
        hist['quick_ep'].append(ep)
        hist['q_w'].append(w); hist['q_d'].append(d); hist['q_l'].append(l)
        print(f'Ep {ep:5d} | KL={mkl:.3f} vErr={mve:.3f} a0 t/p={mct:.1f}/{mcp:.1f} '
              f'| vs {ref["label"]:>5} W{w:2d}D{d:2d}L{l:2d} '
              f'(policy Elo={elos["policy"][ref["label"]]:.0f}) | lr={lr_now:.2e}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 3, figsize=(17, 9))
fig.suptitle('Boop ThompsonZero Training', fontsize=14)

ax = axes[0, 0]
ax.plot(hist['ep'], hist['kl'], label='KL(posterior‖prior)')
ax.set_title('Dirichlet KL loss')
ax.set_xlabel('Episode')
ax2 = ax.twinx()
ax2.plot(hist['ep'], hist['conf_t'], color='tab:purple', alpha=0.6, label='α₀ target')
ax2.plot(hist['ep'], hist['conf_p'], color='tab:brown', alpha=0.6, label='α₀ predicted')
ax2.set_ylabel('mean α₀')
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, fontsize=8)

axes[0, 1].plot(hist['ep'], hist['v_err'], color='tab:orange')
axes[0, 1].set_title('|p_win target − predicted| (legal actions)')
axes[0, 1].set_xlabel('Episode')

# Elo of the model trained-to-episode-N, one line per evaluation track.
def _newest_label(ep):
    return '0' if ep == 0 else str(ep)
for name, _ in EVAL_TRACKS:
    xs, ys = [], []
    for ep, snap in zip(hist['deep_ep'], hist['elo_snap']):
        lbl = _newest_label(ep)
        if lbl in snap[name]:
            xs.append(ep); ys.append(snap[name][lbl])
    axes[0, 2].plot(xs, ys, marker='.', label=name)
axes[0, 2].axhline(START_ELO, color='gray', linestyle='--', linewidth=0.8)
axes[0, 2].set_title('Current-model Elo by track')
axes[0, 2].set_xlabel('Episode'); axes[0, 2].legend(fontsize=8)

# Quick-eval progress pulse: search-free W/D/L vs the most recent benchmark.
axes[1, 0].plot(hist['quick_ep'], hist['q_w'], color='tab:green', marker='.', label='Win')
axes[1, 0].plot(hist['quick_ep'], hist['q_d'], color='gray', linestyle='--', label='Draw')
axes[1, 0].plot(hist['quick_ep'], hist['q_l'], color='tab:red', linestyle=':', label='Loss')
axes[1, 0].set_title(f'Quick eval vs latest benchmark ({QUICK_EVAL_GAMES} games, search-free)')
axes[1, 0].set_xlabel('Episode'); axes[1, 0].legend(fontsize=8)

# Final Elo per benchmark, grouped bars by track.
labels = [b['label'] for b in benchmarks]
x = np.arange(len(labels)); width = 0.8 / max(len(EVAL_TRACKS), 1)
for k, (name, _) in enumerate(EVAL_TRACKS):
    vals = [elos[name].get(l, np.nan) for l in labels]
    axes[1, 1].bar(x + (k - (len(EVAL_TRACKS) - 1) / 2) * width, vals, width, label=name)
axes[1, 1].axhline(START_ELO, color='gray', linestyle='--', linewidth=0.8)
axes[1, 1].set_xticks(x); axes[1, 1].set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
axes[1, 1].set_title('Final Elo per benchmark'); axes[1, 1].set_ylabel('Elo')
axes[1, 1].legend(fontsize=8)

# Head-to-head win-rate matrix for the strongest track (highest sims).
mtrack = EVAL_TRACKS[-1][0]
W = wdl[mtrack]
n = len(labels)
M = np.full((n, n), np.nan)
for i, ri in enumerate(labels):
    for j, cj in enumerate(labels):
        if (ri, cj) in W:
            w, d, l = W[(ri, cj)]; tot = w + d + l
            if tot: M[i, j] = (w + 0.5 * d) / tot
        elif (cj, ri) in W:
            w, d, l = W[(cj, ri)]; tot = w + d + l
            if tot: M[i, j] = 1.0 - (w + 0.5 * d) / tot
im = axes[1, 2].imshow(M, cmap='RdYlGn', vmin=0, vmax=1)
axes[1, 2].set_xticks(range(n)); axes[1, 2].set_yticks(range(n))
axes[1, 2].set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
axes[1, 2].set_yticklabels(labels, fontsize=7)
axes[1, 2].set_title(f'Win-rate matrix — {mtrack} (row vs col)')
for i in range(n):
    for j in range(n):
        if not np.isnan(M[i, j]):
            axes[1, 2].text(j, i, f'{M[i, j]:.2f}', ha='center', va='center',
                            fontsize=6, color='black')
fig.colorbar(im, ax=axes[1, 2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## Arena: ThompsonZero vs the original KataZero agents

The cells below load a checkpoint produced by **`boop_kataboop_training.ipynb`**
(the policy+value `BoopNet`, driven by the original PUCT `BatchedMCTSBot`) and
pit it against this notebook's ThompsonZero net — both **search-free** and
**with search** — alternating colours, with a few random opening plies for
game variety.

- `KATA_CKPT_PATH` accepts either the KataZero run's `latest.pt` (full training
  checkpoint — the `model` entry is used) or any of its `bench_<ep>.pt`
  snapshot files (raw state dicts).
- `KATA_CHANNELS`/`KATA_BLOCKS` must match the architecture that produced the
  checkpoint (the KataZero GPU-branch runs use 128 channels × 8 blocks).
- The ThompsonZero side loads `CHECKPOINT_DIR/latest.pt` if present, otherwise
  it uses the live `base_network` from this session.

In [ ]:
# ── The original KataZero agent: PUCT batched-leaf MCTS on KataNet ────────────
# Verbatim search semantics from boop_kataboop_training.ipynb (virtual loss,
# PUCT, MCTS-Solver backup), so loaded checkpoints play exactly as they were
# trained to.
import math


class _KNode:
    """Lightweight search node with a virtual-loss counter."""
    __slots__ = ['action', 'player', 'prior', 'explore_count',
                 'total_reward', 'vloss', 'outcome', 'children']

    def __init__(self, action, player, prior):
        self.action        = action
        self.player        = player
        self.prior         = prior
        self.explore_count = 0
        self.total_reward  = 0.0
        self.vloss         = 0          # in-flight virtual visits
        self.outcome       = None
        self.children      = []


class KataMCTSBot:
    """The KataZero notebook's BatchedMCTSBot: PUCT selection on policy priors,
    scalar value backups, MCTS-Solver on terminals."""
    def __init__(self, game, network, device, max_simulations,
                 batch_size=16, uct_c=1.4, dirichlet=(0.25, 0.3),
                 solve=True, random_state=None):
        self.game            = game
        self.network         = network
        self.device          = device
        self.max_simulations = max_simulations
        self.batch_size      = batch_size
        self.uct_c           = uct_c
        self.dirichlet       = dirichlet
        self.solve           = solve
        self.max_utility     = game.max_utility()
        self._rng            = random_state or np.random.RandomState()

    def _eval_batch(self, states):
        obs_list = [s.observation_tensor(s.current_player()) for s in states]
        x = batch_to_tensor(obs_list, self.device)
        with torch.no_grad():
            logits, values = self.network(x)
        return logits.cpu().numpy(), values.squeeze(-1).cpu().numpy()

    def _expand(self, node, cur_player, legal, logits, add_noise):
        lg = logits[legal] - logits[legal].max()
        pr = np.exp(lg)
        pr /= pr.sum()
        if add_noise and self.dirichlet is not None:
            eps, alpha = self.dirichlet
            noise = self._rng.dirichlet([alpha] * len(legal))
            pr = (1.0 - eps) * pr + eps * noise
        children = [_KNode(a, cur_player, float(p)) for a, p in zip(legal, pr)]
        self._rng.shuffle(children)
        node.children = children

    def _puct(self, child, parent_n_eff):
        if child.outcome is not None:
            return child.outcome[child.player]
        ec = child.explore_count + child.vloss
        q  = (child.total_reward - child.vloss) / ec if ec > 0 else 0.0
        return q + self.uct_c * child.prior * math.sqrt(parent_n_eff) / (ec + 1)

    def _select_leaf(self, root, root_state):
        path  = [root]
        root.vloss += 1
        state = root_state.clone()
        node  = root
        while node.children and not state.is_terminal():
            parent_n_eff = node.explore_count + node.vloss
            best = max(node.children, key=lambda c: self._puct(c, parent_n_eff))
            state.apply_action(best.action)
            best.vloss += 1
            path.append(best)
            node = best
        return path, state, node

    def _backup_value(self, path, leaf_cur, value):
        for node in reversed(path):
            node.vloss         -= 1
            node.explore_count += 1
            node.total_reward  += value if node.player == leaf_cur else -value

    def _backup_terminal(self, path, returns):
        path[-1].outcome = returns
        solved = self.solve
        for node in reversed(path):
            node.vloss         -= 1
            node.explore_count += 1
            node.total_reward  += returns[node.player]
            if solved and node.children:
                player = node.children[0].player
                best, all_solved = None, True
                for ch in node.children:
                    if ch.outcome is None:
                        all_solved = False
                    elif best is None or ch.outcome[player] > best.outcome[player]:
                        best = ch
                if best is not None and (all_solved or
                                         best.outcome[player] == self.max_utility):
                    node.outcome = best.outcome
                else:
                    solved = False

    def mcts_search(self, state):
        root = _KNode(None, state.current_player(), 1.0)
        logits, values = self._eval_batch([state])
        self._expand(root, state.current_player(), state.legal_actions(),
                     logits[0], add_noise=self.dirichlet is not None)
        root.explore_count = 1
        root.total_reward += float(values[0])

        while root.explore_count < self.max_simulations:
            if root.outcome is not None:
                break
            wave = min(self.batch_size, self.max_simulations - root.explore_count)
            pending = []
            for _ in range(wave):
                if root.outcome is not None:
                    break
                path, st, leaf = self._select_leaf(root, state)
                pending.append((path, leaf, st))

            to_eval = {}
            for path, leaf, st in pending:
                if not st.is_terminal():
                    to_eval[id(leaf)] = (leaf, st)
            results = {}
            if to_eval:
                states = [s for (_, s) in to_eval.values()]
                lg, vv = self._eval_batch(states)
                for (k, (leaf, st)), logit_row, val in zip(to_eval.items(), lg, vv):
                    cur = st.current_player()
                    self._expand(leaf, cur, st.legal_actions(), logit_row, False)
                    results[k] = (cur, float(val))

            for path, leaf, st in pending:
                if st.is_terminal():
                    self._backup_terminal(path, st.returns())
                else:
                    cur, val = results[id(leaf)]
                    self._backup_value(path, cur, val)
        return root


def kata_mcts_move(bot, state):
    """Most-visited action from a KataZero PUCT search."""
    root   = bot.mcts_search(state)
    legal  = state.legal_actions()
    visits = {c.action: c.explore_count for c in root.children}
    counts = np.array([visits.get(a, 0) for a in legal], dtype=np.float32)
    return legal[int(counts.argmax())]


def kata_policy_action(network, state, device):
    """Search-free KataZero move: argmax of the policy head over legal actions."""
    with torch.no_grad():
        logits, _ = network(state_to_tensor(state, device))
    logits = logits.squeeze().cpu().numpy()
    legal  = state.legal_actions()
    return legal[int(logits[legal].argmax())]


# ── Checkpoint loading ────────────────────────────────────────────────────────

def _load_sd(path):
    # Full training checkpoints (latest.pt) also store non-tensor objects — e.g.
    # a numpy float in the LR-scheduler state (np.cos in the LR schedule) — which
    # torch's default weights_only=True unpickler rejects. Try the safe load,
    # then fall back to a full load for these trusted, locally-produced files.
    try:
        sd = torch.load(path, map_location='cpu', weights_only=True)
    except Exception:
        sd = torch.load(path, map_location='cpu', weights_only=False)
    return sd['model'] if isinstance(sd, dict) and 'model' in sd else sd


def load_kata_net(path, channels=128, num_blocks=8):
    net = KataNet(channels, num_blocks)
    net.load_state_dict(_load_sd(path))
    net.eval()
    return net


def load_thompson_net(path, channels, num_blocks):
    net = ThompsonNet(channels, num_blocks)
    net.load_state_dict(_load_sd(path))
    net.eval()
    return net


# ── The arena itself ──────────────────────────────────────────────────────────

def arena(t_net, k_net, game, n_games=40, t_sims=50, k_sims=50,
          device='cpu', opening_plies=4, verbose=True):
    """ThompsonZero (t_net) vs KataZero (k_net). sims == 0 → search-free
    (prior-mean argmax / policy argmax); sims > 0 → full search (posterior-mean
    argmax / most-visited, no root noise). Alternates colours; the first
    `opening_plies` moves of each game are random for variety.
    Returns (wins, draws, losses) from ThompsonZero's perspective."""
    t_bot = (make_thompson_bot(game, t_net, device, t_sims,
                               batch_size=max(1, min(t_sims, 16)))
             if t_sims > 0 else None)
    k_bot = (KataMCTSBot(game, k_net, device, k_sims,
                         batch_size=max(1, min(k_sims, 16)), dirichlet=None)
             if k_sims > 0 else None)
    w = d = l = 0
    for i in range(n_games):
        state = game.new_initial_state()
        t_player = i % 2                 # alternate colours
        ply = 0
        while not state.is_terminal():
            if ply < opening_plies:
                action = random.choice(state.legal_actions())
            elif state.current_player() == t_player:
                action = (_mcts_move(t_bot, state) if t_bot is not None
                          else policy_action(t_net, state, device))
            else:
                action = (kata_mcts_move(k_bot, state) if k_bot is not None
                          else kata_policy_action(k_net, state, device))
            state.apply_action(action)
            ply += 1
        r = state.returns()[t_player]
        if r > 0:   w += 1
        elif r < 0: l += 1
        else:       d += 1
        if verbose and (i + 1) % 10 == 0:
            print(f'    {i + 1:3d}/{n_games}  ThompsonZero W{w} D{d} L{l}')
    return w, d, l


# ── Run it ────────────────────────────────────────────────────────────────────
import os

KATA_CKPT_PATH     = os.path.join('boop_checkpoints_gpu', 'latest.pt')
KATA_CHANNELS      = 128    # must match the checkpoint's architecture
KATA_BLOCKS        = 8
THOMPSON_CKPT_PATH = os.path.join(CHECKPOINT_DIR, 'latest.pt')
ARENA_GAMES        = 40
ARENA_SIMS         = 50     # per-move simulations for the with-search match
ARENA_DEVICE       = 'cpu'  # batch-1 inference: CPU beats DirectML here

if not os.path.exists(KATA_CKPT_PATH):
    print(f'No KataZero checkpoint at {KATA_CKPT_PATH}.')
    print('Train boop_kataboop_training.ipynb first, or point KATA_CKPT_PATH '
          'at one of its bench_<ep>.pt files.')
else:
    kata_net = load_kata_net(KATA_CKPT_PATH, KATA_CHANNELS, KATA_BLOCKS)
    if os.path.exists(THOMPSON_CKPT_PATH):
        thomp_net = load_thompson_net(THOMPSON_CKPT_PATH, CHANNELS, NUM_BLOCKS)
        print(f'ThompsonZero: {THOMPSON_CKPT_PATH}')
    else:
        thomp_net = _cpu_clone(base_network)
        print('ThompsonZero: live network from this session (no checkpoint yet)')
    print(f'KataZero:     {KATA_CKPT_PATH}')

    print(f'\nSearch-free ({ARENA_GAMES} games):')
    w, d, l = arena(thomp_net, kata_net, game, ARENA_GAMES,
                    t_sims=0, k_sims=0, device=ARENA_DEVICE)
    print(f'  ThompsonZero W{w} D{d} L{l}  '
          f'({(w + 0.5 * d) / max(w + d + l, 1):.1%} score)')

    print(f'\n{ARENA_SIMS}-sim MCTS ({ARENA_GAMES} games):')
    w, d, l = arena(thomp_net, kata_net, game, ARENA_GAMES,
                    t_sims=ARENA_SIMS, k_sims=ARENA_SIMS, device=ARENA_DEVICE)
    print(f'  ThompsonZero W{w} D{d} L{l}  '
          f'({(w + 0.5 * d) / max(w + d + l, 1):.1%} score)')